In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
# Load cleaned production dataset

production_df = pd.read_csv(
    "../data/processed/PhiUSIIL_production.csv"
)

print("Dataset shape:", production_df.shape)
print("Missing values:", production_df.isnull().sum().sum())
print("\nLabel distribution:")
print(production_df["label"].value_counts())

Dataset shape: (235370, 19)
Missing values: 0

Label distribution:
label
1    134850
0    100520
Name: count, dtype: int64


In [3]:
# Load cleaned production dataset

production_df = pd.read_csv(
    "../data/processed/PhiUSIIL_production.csv"
)

print("Dataset shape:", production_df.shape)
print("Missing values:", production_df.isnull().sum().sum())
print("\nLabel distribution:")
print(production_df["label"].value_counts())

Dataset shape: (235370, 19)
Missing values: 0

Label distribution:
label
1    134850
0    100520
Name: count, dtype: int64


In [4]:
# Cell 4 - Domain-aware train/holdout split

from urllib.parse import urlparse

def get_domain(url):
    try:
        return urlparse(str(url)).hostname or ""
    except:
        return ""

production_df["DomainGroup"] = production_df["URL"].apply(get_domain)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, holdout_idx = next(
    splitter.split(
        production_df,
        production_df["label"],
        groups=production_df["DomainGroup"]
    )
)

domain_train_df = production_df.iloc[train_idx].copy()
domain_holdout_df = production_df.iloc[holdout_idx].copy()

overlap = set(domain_train_df["DomainGroup"]) & set(domain_holdout_df["DomainGroup"])

print("Training rows:", len(domain_train_df))
print("Holdout rows:", len(domain_holdout_df))
print("Domain overlap:", len(overlap))
print("\nTraining labels:")
print(domain_train_df["label"].value_counts())

Training rows: 187856
Holdout rows: 47514
Domain overlap: 0

Training labels:
label
1    107828
0     80028
Name: count, dtype: int64


In [5]:
# Cell 5 - Legitimate URLs from training split ONLY

legitimate_train_urls = (
    domain_train_df.loc[
        domain_train_df["label"] == 1,
        "URL"
    ]
    .dropna()
    .astype(str)
    .tolist()
)

print("Legitimate training URLs:", len(legitimate_train_urls))

Legitimate training URLs: 107828


In [6]:
# Cell 6 - Generate structurally diverse legitimate URLs

STRUCTURAL_PATHS = [
    "/",

    # Depth 1
    "/about",
    "/contact",
    "/products",
    "/services",
    "/blog",
    "/help",
    "/news",
    "/docs",

    # Depth 2
    "/products/item",
    "/blog/article",
    "/news/latest",
    "/docs/guide",
    "/support/account",
    "/company/about",
    "/resources/articles",

    # Depth 3
    "/products/category/item",
    "/docs/guide/setup",
    "/support/account/settings",
    "/resources/articles/latest",
    "/company/about/team",

    # Depth 4
    "/docs/guide/setup/windows",
    "/products/category/item/details",
    "/support/account/settings/privacy",
]

structural_legitimate_urls = []

for original_url in legitimate_train_urls:

    base_url = original_url.rstrip("/")

    variants = [base_url]

    if "://www." in base_url:
        variants.append(
            base_url.replace("://www.", "://", 1)
        )

    for variant in variants:
        for path in STRUCTURAL_PATHS:
            structural_legitimate_urls.append(
                variant + path
            )

print("Legitimate source URLs:", len(legitimate_train_urls))
print("Structural pool:", len(structural_legitimate_urls))

Legitimate source URLs: 107828
Structural pool: 5175744


In [7]:
# Cell 7 - Controlled 100k augmentation sample

structural_sample = pd.Series(
    structural_legitimate_urls
).sample(
    n=100000,
    random_state=42
).tolist()

print("Total structural pool:", len(structural_legitimate_urls))
print("Sampled for augmentation:", len(structural_sample))

Total structural pool: 5175744
Sampled for augmentation: 100000


In [8]:
# Cell 8 - Use the same feature extractor as the application

import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.src.feature_extractor import extract_url_features

print("Feature extractor imported successfully.")

Feature extractor imported successfully.


In [9]:
# Cell 9 - Extract features from augmented legitimate URLs

augmented_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in structural_sample
    ]
)

augmented_features_df.insert(
    0,
    "URL",
    structural_sample
)

augmented_features_df["label"] = 1

print("Augmented shape:", augmented_features_df.shape)

print("\nNoOfSlashes:")
print(
    augmented_features_df["NoOfSlashes"]
    .value_counts()
    .sort_index()
)

print("\nPathDepth:")
print(
    augmented_features_df["PathDepth"]
    .value_counts()
    .sort_index()
)

Augmented shape: (100000, 19)

NoOfSlashes:
NoOfSlashes
3    37379
4    29139
5    20942
6    12540
Name: count, dtype: int64

PathDepth:
PathDepth
0     4199
1    33180
2    29139
3    20942
4    12540
Name: count, dtype: int64


In [11]:
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth"
]

print("Feature count:", len(ROBUSTNESS_FEATURES))

Feature count: 17


In [12]:
# Cell 10 - Final augmented training dataset

final_train_df = pd.concat(
    [
        domain_train_df,
        augmented_features_df
    ],
    ignore_index=True
)

X_train_final = final_train_df[ROBUSTNESS_FEATURES].copy()
y_train_final = final_train_df["label"].copy()

X_holdout = domain_holdout_df[ROBUSTNESS_FEATURES].copy()
y_holdout = domain_holdout_df["label"].copy()

print("Final training rows:", len(final_train_df))
print("Holdout rows:", len(domain_holdout_df))
print("Feature count:", X_train_final.shape[1])
print("Missing training values:", X_train_final.isnull().sum().sum())

print("\nFinal training labels:")
print(y_train_final.value_counts())

Final training rows: 287856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

Final training labels:
label
1    207828
0     80028
Name: count, dtype: int64


In [ ]:
print("ROBUSTNESS_FEATURES exists:", "ROBUSTNESS_FEATURES" in globals())

ROBUSTNESS_FEATURES exists: False


In [13]:
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth"
]

print("Feature count:", len(ROBUSTNESS_FEATURES))

Feature count: 17


In [14]:
# Cell 11 - Train structural-fix candidate model

tld_encoder = TargetEncoder(
    target_type="binary",
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tld_target_encoder",
            tld_encoder,
            ["TLD"]
        ),
        (
            "numeric",
            "passthrough",
            [
                feature
                for feature in ROBUSTNESS_FEATURES
                if feature != "TLD"
            ]
        )
    ],
    remainder="drop"
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

structural_candidate_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

structural_candidate_pipeline.fit(
    X_train_final,
    y_train_final
)

print("Candidate model trained successfully.")
print("Classes:", structural_candidate_pipeline.classes_)

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Candidate model trained successfully.
Classes: [0 1]


In [19]:
print("Candidate model classes:", structural_candidate_pipeline.classes_)

print("\nTraining label distribution:")
print(y_train_final.value_counts())

print("\nCandidate test:")
test_candidate_url("https://www.google.com")
test_candidate_url("https://www.microsoft.com")
test_candidate_url("http://secure-login-example.com/verify/account")
test_candidate_url("https://account-verification-update.com/login")

Candidate model classes: [0 1]

Training label distribution:
label
1    207828
0     80028
Name: count, dtype: int64

Candidate test:

URL: https://www.google.com
Prediction: Legitimate
Legitimate: 99.43% | Phishing: 0.57%

URL: https://www.microsoft.com
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: http://secure-login-example.com/verify/account
Prediction: Phishing
Legitimate: 3.67% | Phishing: 96.33%

URL: https://account-verification-update.com/login
Prediction: Phishing
Legitimate: 33.33% | Phishing: 66.67%


In [20]:
diagnostic_urls = [
    "https://www.google.com",
    "https://www.google.com/search?q=hello%20world&page=2",
    "https://www.google.com/location?lat=19.1922063&lng=72.8777",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "http://secure-login-example.com/verify/account",
]

for url in diagnostic_urls:
    features = extract_url_features(url)

    print("\n" + "=" * 80)
    print(url)

    for feature in ROBUSTNESS_FEATURES:
        print(f"{feature}: {features[feature]}")


https://www.google.com
URLLength: 22
DomainLength: 14
IsDomainIP: 0
TLD: com
TLDLength: 3
NoOfSubDomain: 1
HasObfuscation: 0
NoOfObfuscatedChar: 0
NoOfDegitsInURL: 0
NoOfEqualsInURL: 0
NoOfQMarkInURL: 0
IsHTTPS: 1
NoOfDots: 2
NoOfSlashes: 2
SuspiciousKeywordCount: 0
HasHyphenInDomain: 0
PathDepth: 0

https://www.google.com/search?q=hello%20world&page=2
URLLength: 52
DomainLength: 14
IsDomainIP: 0
TLD: com
TLDLength: 3
NoOfSubDomain: 1
HasObfuscation: 1
NoOfObfuscatedChar: 3
NoOfDegitsInURL: 3
NoOfEqualsInURL: 2
NoOfQMarkInURL: 1
IsHTTPS: 1
NoOfDots: 2
NoOfSlashes: 3
SuspiciousKeywordCount: 0
HasHyphenInDomain: 0
PathDepth: 1

https://www.google.com/location?lat=19.1922063&lng=72.8777
URLLength: 58
DomainLength: 14
IsDomainIP: 0
TLD: com
TLDLength: 3
NoOfSubDomain: 1
HasObfuscation: 0
NoOfObfuscatedChar: 0
NoOfDegitsInURL: 15
NoOfEqualsInURL: 2
NoOfQMarkInURL: 1
IsHTTPS: 1
NoOfDots: 4
NoOfSlashes: 3
SuspiciousKeywordCount: 0
HasHyphenInDomain: 0
PathDepth: 1

https://www.google.com/map

In [21]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

candidate_holdout_pred = structural_candidate_pipeline.predict(X_holdout)

print("Candidate Holdout Accuracy:")
print(accuracy_score(y_holdout, candidate_holdout_pred))

print("\nCandidate Holdout Classification Report:")
print(
    classification_report(
        y_holdout,
        candidate_holdout_pred,
        target_names=["Phishing", "Legitimate"]
    )
)

print("\nCandidate Confusion Matrix:")
print(confusion_matrix(y_holdout, candidate_holdout_pred))

Candidate Holdout Accuracy:
0.9857094751020752

Candidate Holdout Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      0.97      0.98     20492
  Legitimate       0.98      1.00      0.99     27022

    accuracy                           0.99     47514
   macro avg       0.99      0.98      0.99     47514
weighted avg       0.99      0.99      0.99     47514


Candidate Confusion Matrix:
[[19877   615]
 [   64 26958]]


In [22]:
comparison_urls = [
    "https://www.google.com/search?q=hello%20world&page=2",
    "https://www.google.com/location?lat=19.1922063&lng=72.8777",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser",
]

print("V4 vs CANDIDATE")
print("=" * 80)

for url in comparison_urls:
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    # V4
    v4_probs = structural_candidate_pipeline_v4.predict_proba(input_df)[0]
    v4_classes = list(structural_candidate_pipeline_v4.classes_)
    v4_phishing = v4_probs[v4_classes.index(0)]

    # Candidate
    cand_probs = structural_candidate_pipeline.predict_proba(input_df)[0]
    cand_classes = list(structural_candidate_pipeline.classes_)
    cand_phishing = cand_probs[cand_classes.index(0)]

    print(f"\n{url}")
    print(f"V4 phishing:        {v4_phishing:.2%}")
    print(f"Candidate phishing: {cand_phishing:.2%}")

V4 vs CANDIDATE


NameError: name 'structural_candidate_pipeline_v4' is not defined

In [23]:
for url in comparison_urls:
    print("\n" + "=" * 80)
    print(url)
    test_candidate_url(url)


https://www.google.com/search?q=hello%20world&page=2

URL: https://www.google.com/search?q=hello%20world&page=2
Prediction: Phishing
Legitimate: 0.67% | Phishing: 99.33%

https://www.google.com/location?lat=19.1922063&lng=72.8777

URL: https://www.google.com/location?lat=19.1922063&lng=72.8777
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%

https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%

https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%

https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser
Prediction: Phishing
Legitimate: 0.00% | Phish

In [24]:
print("CANDIDATE TRAINING FEATURE RANGES")
print("=" * 60)

for feature in [
    "URLLength",
    "NoOfDots",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfSlashes",
    "NoOfDegitsInURL",
    "PathDepth",
]:
    print(f"\n{feature}")
    print(
        final_train_df.groupby("label")[feature]
        .agg(["min", "mean", "median", "max"])
    )

CANDIDATE TRAINING FEATURE RANGES

URLLength
       min       mean  median   max
label                              
0       14  45.874144    34.0  6097
1       14  33.310488    31.0    89

NoOfDots
       min      mean  median  max
label                            
0        1  2.386815     2.0   99
1        1  1.922075     2.0    5

NoOfEqualsInURL
       min      mean  median  max
label                            
0        0  0.136452     0.0  176
1        0  0.000000     0.0    0

NoOfQMarkInURL
       min     mean  median  max
label                           
0        0  0.06489     0.0    4
1        0  0.00000     0.0    0

NoOfSlashes
       min      mean  median  max
label                            
0        2  3.002374     3.0   68
1        2  3.003922     2.0    6

NoOfDegitsInURL
       min      mean  median   max
label                             
0        0  4.338494     0.0  2011
1        0  0.050792     0.0     8

PathDepth
       min      mean  median  max
label        

In [17]:
def test_candidate_url(url):
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline.predict(input_df)[0]
    probabilities = structural_candidate_pipeline.predict_proba(input_df)[0]

    classes = list(structural_candidate_pipeline.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    prediction_label = (
        "Phishing"
        if prediction == 0
        else "Legitimate"
    )

    print(f"\nURL: {url}")
    print(f"Prediction: {prediction_label}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )

In [18]:
candidate_test_urls = [
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser",

    "https://www.google.com/search?q=hello%20world&page=2",
    "https://www.google.com/location?lat=19.1922063&lng=72.8777",
]

print("NEW CANDIDATE MODEL TESTS")

for url in candidate_test_urls:
    test_candidate_url(url)

NEW CANDIDATE MODEL TESTS

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%

URL: https://www.google.com/search?q=hello%20world&page=2
Prediction: Phishing
Legitimate: 0.67% | Phishing: 99.33%

URL: https://www.google.com/location?lat=19.1922063&lng=72.8777
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%


In [ ]:
print("ROBUSTNESS_FEATURES exists:", "ROBUSTNESS_FEATURES" in globals())

ROBUSTNESS_FEATURES exists: False


In [ ]:
# Cell 12 - Evaluate on untouched domain-aware holdout

holdout_predictions = structural_candidate_pipeline.predict(
    X_holdout
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_holdout,
            holdout_predictions
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_holdout,
        holdout_predictions
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_holdout,
        holdout_predictions
    )
)

Accuracy: 0.9857

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     20492
           1       0.98      1.00      0.99     27022

    accuracy                           0.99     47514
   macro avg       0.99      0.98      0.99     47514
weighted avg       0.99      0.99      0.99     47514

Confusion Matrix:
[[19877   615]
 [   64 26958]]


In [ ]:
# Cell 13 - Legitimate URL sanity test

legitimate_test_urls = [
    "https://www.google.com",
    "https://www.google.com/maps",
    "https://www.google.com/maps/place",
    "https://www.google.com/a/b",
    "https://www.google.com/a/b/c",
    "https://www.paypal.com",
    "https://www.paypal.com/home",
    "https://www.paypal.com/in/home",
]

classes = list(structural_candidate_pipeline.classes_)

phishing_index = classes.index(0)
legitimate_index = classes.index(1)

results = []

for url in legitimate_test_urls:

    features = extract_url_features(url)

    input_df = pd.DataFrame(
        [features]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline.predict(
        input_df
    )[0]

    probabilities = structural_candidate_pipeline.predict_proba(
        input_df
    )[0]

    results.append({
        "URL": url,
        "Prediction": (
            "Legitimate"
            if prediction == 1
            else "Phishing"
        ),
        "Legitimate %": round(
            probabilities[legitimate_index] * 100,
            2
        ),
        "Phishing Risk %": round(
            probabilities[phishing_index] * 100,
            2
        ),
    })

pd.DataFrame(results)

,URL,Prediction,Legitimate %,Phishing Risk %
0,https://www.google.com,Legitimate,99.43,0.57
1,https://www.google.com/maps,Legitimate,100.00,0.00
2,https://www.google.com/maps/place,Legitimate,100.00,0.00
3,https://www.google.com/a/b,Legitimate,79.00,21.00
4,https://www.google.com/a/b/c,Legitimate,63.33,36.67
5,https://www.paypal.com,Legitimate,99.43,0.57
6,https://www.paypal.com/home,Legitimate,100.00,0.00
7,https://www.paypal.com/in/home,Legitimate,87.33,12.67


In [ ]:
# Cell 14 - Phishing-like URL sanity test

phishing_test_urls = [
    "https://paypal-login-security.com",
    "https://account-verification-update.com/login",
    "https://secure-bank-login.xyz/account",
    "https://verify-your-account.com/update",
    "https://login-security-check.net/verify",
]

results = []

for url in phishing_test_urls:

    features = extract_url_features(url)

    input_df = pd.DataFrame(
        [features]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline.predict(
        input_df
    )[0]

    probabilities = structural_candidate_pipeline.predict_proba(
        input_df
    )[0]

    results.append({
        "URL": url,
        "Prediction": (
            "Legitimate"
            if prediction == 1
            else "Phishing"
        ),
        "Legitimate %": round(
            probabilities[legitimate_index] * 100,
            2
        ),
        "Phishing Risk %": round(
            probabilities[phishing_index] * 100,
            2
        ),
    })

pd.DataFrame(results)

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\Tr

,URL,Prediction,Legitimate %,Phishing Risk %
0,https://paypal-login-security.com,Phishing,1.72,98.28
1,https://account-verification-update.com/login,Phishing,33.33,66.67
2,https://secure-bank-login.xyz/account,Phishing,3.00,97.00
3,https://verify-your-account.com/update,Phishing,18.00,82.00
4,https://login-security-check.net/verify,Phishing,13.67,86.33


In [ ]:
# Cell 15 - Save validated production model safely

from pathlib import Path
import shutil
import joblib

model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

backup_path = Path(
    "../models/trustlens_production_pipeline_backup.joblib"
)

# Preserve existing production model if backup does not already exist
if model_path.exists() and not backup_path.exists():
    shutil.copy2(model_path, backup_path)
    print("Backup created.")
elif backup_path.exists():
    print("Backup already exists - leaving it untouched.")

# Save validated structural-fix model
joblib.dump(
    structural_candidate_pipeline,
    model_path
)

print("New production model saved successfully.")
print("Saved to:", model_path)

Backup already exists - leaving it untouched.
New production model saved successfully.
Saved to: ..\models\trustlens_production_pipeline.joblib


In [ ]:
# Cell 16 - Final reload verification

reloaded_pipeline = joblib.load(
    "../models/trustlens_production_pipeline.joblib"
)

print("Production model reloaded successfully.")
print("Classes:", reloaded_pipeline.classes_)

test_url = "https://www.google.com/maps/place"

features = extract_url_features(test_url)
input_df = pd.DataFrame([features])[ROBUSTNESS_FEATURES]

prediction = reloaded_pipeline.predict(input_df)[0]
probabilities = reloaded_pipeline.predict_proba(input_df)[0]

classes = list(reloaded_pipeline.classes_)
phishing_index = classes.index(0)

print("Test URL:", test_url)
print(
    "Prediction:",
    "Legitimate" if prediction == 1 else "Phishing"
)
print(
    "Phishing Risk:",
    round(probabilities[phishing_index] * 100, 2),
    "%"
)

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


Production model reloaded successfully.
Classes: [0 1]


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\Tr

Test URL: https://www.google.com/maps/place
Prediction: Legitimate
Phishing Risk: 0.0 %


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\Tr

In [ ]:
print("\nPathDepth:")
print(
    augmented_features_df["PathDepth"]
    .value_counts()
    .sort_index()
)


PathDepth:
PathDepth
0     4199
1    33180
2    29139
3    20942
4    12540
Name: count, dtype: int64


In [ ]:
# =========================================================
# QUERY-HEAVY LEGITIMATE URL AUGMENTATION
# =========================================================

import random
from urllib.parse import urlsplit, urlunsplit

random.seed(42)

QUERY_TEMPLATES = [
    "/search?q=example",
    "/search?q=example&page=2",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.9234965",
    "/maps/@19.1922063,72.9234965,11z",
    "/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
    "/redirect?next=%2Faccount%2Fsettings",
    "/search?q=hello%20world",
    "/docs?page=2&section=installation",
    "/view?item=123456&source=web&lang=en",
]

query_legitimate_urls = []

for original_url in legitimate_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        # IMPORTANT:
        # use only scheme + legitimate training hostname
        # instead of appending onto the original path/query
        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                "",
                "",
                "",
            )
        ).rstrip("/")

        variants = [base_url]

        # www/non-www structural variation
        if "://www." in base_url:
            variants.append(
                base_url.replace(
                    "://www.",
                    "://",
                    1,
                )
            )

        for variant in variants:

            for template in QUERY_TEMPLATES:

                query_legitimate_urls.append(
                    variant + template
                )

    except Exception:
        continue


print(
    "Query-heavy legitimate pool:",
    len(query_legitimate_urls)
)

Query-heavy legitimate pool: 3234840


In [26]:
# =========================================================
# SAMPLE QUERY-HEAVY LEGITIMATE URLS
# =========================================================

QUERY_AUGMENTATION_SIZE = 50_000

query_augmented_urls = random.sample(
    query_legitimate_urls,
    k=min(
        QUERY_AUGMENTATION_SIZE,
        len(query_legitimate_urls),
    ),
)

print(
    "Query-heavy URLs sampled:",
    len(query_augmented_urls),
)

print("\nExamples:")

for url in query_augmented_urls[:10]:
    print(url)

NameError: name 'random' is not defined

In [25]:
# =========================================================
# EXTRACT FEATURES FROM QUERY-HEAVY LEGITIMATE URLS
# =========================================================

# Exact 17 production features
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]


# Extract the same features used by TrustLens
query_augmented_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in query_augmented_urls
    ]
)

# Add original URL for inspection
query_augmented_features_df.insert(
    0,
    "URL",
    query_augmented_urls,
)

# All generated URLs are legitimate examples
query_augmented_features_df["label"] = 1


# =========================================================
# BASIC CHECKS
# =========================================================

print(
    "Query augmentation shape:",
    query_augmented_features_df.shape,
)

print(
    "Feature count:",
    len(ROBUSTNESS_FEATURES),
)

print(
    "Missing values:",
    query_augmented_features_df[
        ROBUSTNESS_FEATURES
    ].isnull().sum().sum(),
)


# =========================================================
# DISTRIBUTION CHECKS
# =========================================================

print("\nURL Length:")
print(
    query_augmented_features_df[
        "URLLength"
    ].describe()
)

print("\nDigits:")
print(
    query_augmented_features_df[
        "NoOfDegitsInURL"
    ].describe()
)

print("\nEquals signs:")
print(
    query_augmented_features_df[
        "NoOfEqualsInURL"
    ]
    .value_counts()
    .sort_index()
)

print("\nQuestion marks:")
print(
    query_augmented_features_df[
        "NoOfQMarkInURL"
    ]
    .value_counts()
    .sort_index()
)

print("\nObfuscation:")
print(
    query_augmented_features_df[
        "HasObfuscation"
    ]
    .value_counts()
    .sort_index()
)

NameError: name 'query_augmented_urls' is not defined

In [ ]:
# =========================================================
# FINAL TRAINING DATA
# Original + Path Augmentation + Query Augmentation
# =========================================================

final_train_df_v2 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        query_augmented_features_df,
    ],
    ignore_index=True,
)


X_train_final_v2 = final_train_df_v2[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v2 = final_train_df_v2[
    "label"
].copy()


# Holdout remains completely untouched
X_holdout_v2 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v2 = domain_holdout_df[
    "label"
].copy()


print(
    "Final training rows:",
    len(final_train_df_v2)
)

print(
    "Holdout rows:",
    len(domain_holdout_df)
)

print(
    "Feature count:",
    X_train_final_v2.shape[1]
)

print(
    "Missing training values:",
    X_train_final_v2.isnull().sum().sum()
)

print("\nFinal training labels:")
print(
    y_train_final_v2.value_counts()
)

Final training rows: 337856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

Final training labels:
label
1    257828
0     80028
Name: count, dtype: int64


In [ ]:
# =========================================================
# TRAIN V2 CANDIDATE MODEL
# DO NOT SAVE TO PRODUCTION YET
# =========================================================

categorical_features_v2 = ["TLD"]

numeric_features_v2 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]

preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v2,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v2,
        ),
    ]
)

candidate_model_v2 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", candidate_model_v2),
    ]
)


print("Training V2 candidate...")

structural_candidate_pipeline_v2.fit(
    X_train_final_v2,
    y_train_final_v2,
)

print("V2 candidate training complete.")

Training V2 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V2 candidate training complete.


In [ ]:
# =========================================================
# EVALUATE V2 CANDIDATE ON UNTOUCHED HOLDOUT
# =========================================================

holdout_predictions_v2 = structural_candidate_pipeline_v2.predict(
    X_holdout_v2
)

holdout_accuracy_v2 = accuracy_score(
    y_holdout_v2,
    holdout_predictions_v2,
)

print(
    "V2 Holdout Accuracy:",
    round(holdout_accuracy_v2, 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_holdout_v2,
        holdout_predictions_v2,
        digits=4,
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v2,
        holdout_predictions_v2,
    )
)

V2 Holdout Accuracy: 0.9853

Classification Report:
              precision    recall  f1-score   support

           0     0.9951    0.9706    0.9827     20492
           1     0.9781    0.9964    0.9872     27022

    accuracy                         0.9853     47514
   macro avg     0.9866    0.9835    0.9850     47514
weighted avg     0.9855    0.9853    0.9853     47514


Confusion Matrix:
[[19890   602]
 [   97 26925]]


In [ ]:
# =========================================================
# V2 LEGITIMATE URL SANITY TESTS
# =========================================================

legitimate_test_urls_v2 = [
    # Original simple/path tests
    "https://www.google.com",
    "https://www.google.com/maps",
    "https://www.google.com/maps/place",
    "https://www.google.com/a/b",
    "https://www.google.com/a/b/c",

    "https://www.paypal.com",
    "https://www.paypal.com/home",
    "https://www.paypal.com/in/home",

    # Real long Google Maps URL that previously failed
    "https://www.google.com/maps/@19.1922063,72.9234965,11.82z?entry=ttu&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D",

    # Generic legitimate-looking long structures
    "https://example.com/search?q=hello%20world&page=2",
    "https://example.com/location?lat=19.1922063&lng=72.9234965",
    "https://example.com/docs/view?id=12345&lang=en",
]


def test_candidate_url_v2(url):

    features = pd.DataFrame(
        [extract_url_features(url)]
    )[ROBUSTNESS_FEATURES]

    probabilities = (
        structural_candidate_pipeline_v2
        .predict_proba(features)[0]
    )

    classes = (
        structural_candidate_pipeline_v2
        .classes_
    )

    probability_map = dict(
        zip(classes, probabilities)
    )

    phishing_risk = (
        probability_map.get(0, 0.0) * 100
    )

    legitimate_probability = (
        probability_map.get(1, 0.0) * 100
    )

    prediction = (
        "Legitimate"
        if legitimate_probability >= phishing_risk
        else "Phishing"
    )

    print(url)
    print(
        f"Prediction: {prediction} | "
        f"Legit: {legitimate_probability:.2f}% | "
        f"Phishing risk: {phishing_risk:.2f}%"
    )
    print("-" * 90)


for url in legitimate_test_urls_v2:
    test_candidate_url_v2(url)

https://www.google.com
Prediction: Legitimate | Legit: 100.00% | Phishing risk: 0.00%
------------------------------------------------------------------------------------------
https://www.google.com/maps
Prediction: Legitimate | Legit: 100.00% | Phishing risk: 0.00%
------------------------------------------------------------------------------------------
https://www.google.com/maps/place
Prediction: Legitimate | Legit: 99.67% | Phishing risk: 0.33%
------------------------------------------------------------------------------------------
https://www.google.com/a/b
Prediction: Legitimate | Legit: 81.00% | Phishing risk: 19.00%
------------------------------------------------------------------------------------------
https://www.google.com/a/b/c
Prediction: Legitimate | Legit: 63.00% | Phishing risk: 37.00%
------------------------------------------------------------------------------------------
https://www.paypal.com
Prediction: Legitimate | Legit: 100.00% | Phishing risk: 0.00%
----

In [ ]:
# =========================================================
# DIAGNOSE THE REAL LONG LEGITIMATE URL
# =========================================================

problem_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

problem_features = extract_url_features(problem_url)

print("Problem URL length:", len(problem_url))
print("\nProblem URL features:\n")

for feature in ROBUSTNESS_FEATURES:
    print(
        f"{feature:28} : "
        f"{problem_features[feature]}"
    )


print("\n" + "=" * 60)
print("QUERY AUGMENTATION COMPARISON")
print("=" * 60)

numeric_compare_features = [
    "URLLength",
    "DomainLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

for feature in numeric_compare_features:

    series = query_augmented_features_df[feature]

    print(f"\n{feature}")
    print(
        "Problem:",
        problem_features[feature],
        "| Aug mean:",
        round(series.mean(), 2),
        "| 95%:",
        round(series.quantile(0.95), 2),
        "| Max:",
        series.max(),
    )

Problem URL length: 109

Problem URL features:

URLLength                    : 109
DomainLength                 : 14
IsDomainIP                   : 0
TLD                          : com
TLDLength                    : 3
NoOfSubDomain                : 1
HasObfuscation               : 1
NoOfObfuscatedChar           : 6
NoOfDegitsInURL              : 26
NoOfEqualsInURL              : 2
NoOfQMarkInURL               : 1
IsHTTPS                      : 1
NoOfDots                     : 5
NoOfSlashes                  : 4
SuspiciousKeywordCount       : 0
HasHyphenInDomain            : 0
PathDepth                    : 2

QUERY AUGMENTATION COMPARISON

URLLength
Problem: 109 | Aug mean: 55.88 | 95%: 75.0 | Max: 98

DomainLength
Problem: 14 | Aug mean: 17.23 | 95%: 26.0 | Max: 45

NoOfSubDomain
Problem: 1 | Aug mean: 0.66 | 95%: 2.0 | Max: 4

HasObfuscation
Problem: 1 | Aug mean: 0.13 | 95%: 1.0 | Max: 1

NoOfObfuscatedChar
Problem: 6 | Aug mean: 0.6 | 95%: 6.0 | Max: 6

NoOfDegitsInURL
Problem: 26 |

In [ ]:
# =========================================================
# V3 - BETTER QUERY AUGMENTATION
# Normal + Long/Complex Legitimate URL Structures
# =========================================================

import random
from urllib.parse import urlsplit, urlunsplit

random.seed(42)


NORMAL_QUERY_TEMPLATES = [
    "/search?q=example",
    "/search?q=example&page=2",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.9234965",
    "/search?q=hello%20world",
    "/redirect?next=%2Faccount%2Fsettings",
]


COMPLEX_QUERY_TEMPLATES = [
    # Coordinates + queries
    "/location/@19.1922063,72.9234965,11.82z"
    "?entry=web&mode=view",

    "/location/@19.1922063,72.9234965,11.82z"
    "?entry=web&source=browser&mode=view",

    # Coordinates + encoded parameter
    "/location/@19.1922063,72.9234965,11.82z"
    "?entry=web&data=ExampleLongValue1234567890%3D%3D",

    "/location/@19.1922063,72.9234965,11.82z"
    "?source=browser&data=AbCdEf12345678901234567890%3D%3D",

    # Long IDs / tokens
    "/view/item/1234567890"
    "?session=ABCDEF12345678901234567890&lang=en",

    "/docs/view"
    "?id=12345678901234567890"
    "&source=browser"
    "&lang=en",

    # Encoded values
    "/redirect"
    "?next=%2Faccount%2Fsettings%2Fprivacy"
    "&source=web",

    "/search"
    "?q=hello%20world"
    "&category=articles"
    "&page=123456",

    # Decimal-heavy structures
    "/data/@40.712776,-74.005974,12.50z"
    "?view=detail&source=web",

    "/place/@51.507351,-0.127758,14.25z"
    "?entry=browser"
    "&data=Example1234567890%3D%3D",
]


better_query_pool = []

for original_url in legitimate_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                "",
                "",
                "",
            )
        ).rstrip("/")

        variants = [base_url]

        if "://www." in base_url:
            variants.append(
                base_url.replace(
                    "://www.",
                    "://",
                    1,
                )
            )

        for variant in variants:

            for template in NORMAL_QUERY_TEMPLATES:
                better_query_pool.append(
                    variant + template
                )

            for template in COMPLEX_QUERY_TEMPLATES:
                better_query_pool.append(
                    variant + template
                )

    except Exception:
        continue


print(
    "Better query pool:",
    len(better_query_pool)
)

Better query pool: 4313120


In [ ]:
# =========================================================
# V3 - STRATIFIED QUERY AUGMENTATION SAMPLE
# 25k normal + 25k complex
# =========================================================

normal_query_pool = []
complex_query_pool = []

for original_url in legitimate_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                "",
                "",
                "",
            )
        ).rstrip("/")

        variants = [base_url]

        if "://www." in base_url:
            variants.append(
                base_url.replace(
                    "://www.",
                    "://",
                    1,
                )
            )

        for variant in variants:

            for template in NORMAL_QUERY_TEMPLATES:
                normal_query_pool.append(
                    variant + template
                )

            for template in COMPLEX_QUERY_TEMPLATES:
                complex_query_pool.append(
                    variant + template
                )

    except Exception:
        continue


NORMAL_SAMPLE_SIZE = 25_000
COMPLEX_SAMPLE_SIZE = 25_000

normal_query_sample = random.sample(
    normal_query_pool,
    k=min(
        NORMAL_SAMPLE_SIZE,
        len(normal_query_pool),
    ),
)

complex_query_sample = random.sample(
    complex_query_pool,
    k=min(
        COMPLEX_SAMPLE_SIZE,
        len(complex_query_pool),
    ),
)

better_query_sample = (
    normal_query_sample
    + complex_query_sample
)

random.shuffle(
    better_query_sample
)


print(
    "Normal query pool:",
    len(normal_query_pool)
)

print(
    "Complex query pool:",
    len(complex_query_pool)
)

print(
    "Normal sampled:",
    len(normal_query_sample)
)

print(
    "Complex sampled:",
    len(complex_query_sample)
)

print(
    "Total V3 query sample:",
    len(better_query_sample)
)

print("\nExamples:")

for url in better_query_sample[:10]:
    print(url)

Normal query pool: 2156560
Complex query pool: 2156560
Normal sampled: 25000
Complex sampled: 25000
Total V3 query sample: 50000

Examples:
https://www.maroc-hebdo.press.ma/search?q=hello%20world
https://koama.es/search?q=example&page=2
https://wolfgangdigital.com/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view
https://www.tshirtsthatsuck.com/products?id=12345
https://www.existentialcomics.com/results?query=example&sort=recent
https://www.neoformix.com/search?q=example&page=2
https://www.lugaresdenieve.com/docs/view?id=12345678901234567890&source=browser&lang=en
https://elgincounty.ca/location/@19.1922063,72.9234965,11.82z?source=browser&data=AbCdEf12345678901234567890%3D%3D
https://orient-watch.jp/search?q=example
https://www.elisgeo.com/location/@19.1922063,72.9234965,11.82z?source=browser&data=AbCdEf12345678901234567890%3D%3D


In [ ]:
# =========================================================
# V3 - EXTRACT FEATURES + COVERAGE CHECK
# =========================================================

better_query_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in better_query_sample
    ]
)

better_query_features_df.insert(
    0,
    "URL",
    better_query_sample
)

better_query_features_df["label"] = 1


print(
    "V3 query augmentation shape:",
    better_query_features_df.shape
)

print(
    "Missing values:",
    better_query_features_df[
        ROBUSTNESS_FEATURES
    ].isnull().sum().sum()
)


print("\nURL Length:")
print(
    better_query_features_df[
        "URLLength"
    ].describe()
)

print("\nDigits:")
print(
    better_query_features_df[
        "NoOfDegitsInURL"
    ].describe()
)

print("\nObfuscated characters:")
print(
    better_query_features_df[
        "NoOfObfuscatedChar"
    ].describe()
)

print("\nDots:")
print(
    better_query_features_df[
        "NoOfDots"
    ].describe()
)


print("\n" + "=" * 60)
print("PROBLEM URL VS V3 COVERAGE")
print("=" * 60)

coverage_features = [
    "URLLength",
    "NoOfDegitsInURL",
    "NoOfObfuscatedChar",
    "NoOfDots",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfSlashes",
    "PathDepth",
]

for feature in coverage_features:

    series = better_query_features_df[feature]

    print(f"\n{feature}")
    print(
        "Problem:",
        problem_features[feature],
        "| V3 mean:",
        round(series.mean(), 2),
        "| V3 95%:",
        round(series.quantile(0.95), 2),
        "| V3 max:",
        series.max(),
    )

V3 query augmentation shape: (50000, 19)
Missing values: 0

URL Length:
count    50000.000000
mean        72.385700
std         23.858806
min         31.000000
25%         51.000000
50%         72.000000
75%         89.000000
max        139.000000
Name: URLLength, dtype: float64

Digits:
count    50000.000000
mean        13.726940
std         13.246372
min          0.000000
25%          2.000000
50%          8.000000
75%         22.000000
max         50.000000
Name: NoOfDegitsInURL, dtype: float64

Obfuscated characters:
count    50000.00000
mean         1.95810
std          2.88665
min          0.00000
25%          0.00000
50%          0.00000
75%          6.00000
max          9.00000
Name: NoOfObfuscatedChar, dtype: float64

Dots:
count    50000.000000
mean         2.665120
std          1.522124
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max          8.000000
Name: NoOfDots, dtype: float64

PROBLEM URL VS V3 COVERAGE

URLLength
Problem: 10

In [ ]:
# =========================================================
# BUILD V3 FINAL TRAINING DATA
# Original + 100k Path Augmentation + 50k Better Query Augmentation
# =========================================================

final_train_df_v3 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
    ],
    ignore_index=True,
)


X_train_final_v3 = final_train_df_v3[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v3 = final_train_df_v3[
    "label"
].copy()


# SAME untouched holdout
X_holdout_v3 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v3 = domain_holdout_df[
    "label"
].copy()


print(
    "V3 training rows:",
    len(final_train_df_v3)
)

print(
    "Holdout rows:",
    len(domain_holdout_df)
)

print(
    "Feature count:",
    X_train_final_v3.shape[1]
)

print(
    "Missing training values:",
    X_train_final_v3.isnull().sum().sum()
)

print("\nV3 training labels:")
print(
    y_train_final_v3.value_counts()
)

V3 training rows: 337856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

V3 training labels:
label
1    257828
0     80028
Name: count, dtype: int64


In [ ]:
# =========================================================
# TRAIN V3 CANDIDATE MODEL
# DO NOT SAVE TO PRODUCTION YET
# =========================================================

categorical_features_v3 = ["TLD"]

numeric_features_v3 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]

preprocessor_v3 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v3,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v3,
        ),
    ]
)

candidate_model_v3 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v3 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v3),
        ("model", candidate_model_v3),
    ]
)

print("Training V3 candidate...")

structural_candidate_pipeline_v3.fit(
    X_train_final_v3,
    y_train_final_v3,
)

print("V3 candidate training complete.")

Training V3 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V3 candidate training complete.


In [ ]:
# =========================================================
# V3 - HOLDOUT EVALUATION
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

y_pred_v3 = structural_candidate_pipeline_v3.predict(
    X_holdout_v3
)

accuracy_v3 = accuracy_score(
    y_holdout_v3,
    y_pred_v3,
)

print(
    "V3 Holdout Accuracy:",
    round(accuracy_v3, 4)
)

print("\nClassification Report:\n")
print(
    classification_report(
        y_holdout_v3,
        y_pred_v3,
        digits=4,
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v3,
        y_pred_v3,
    )
)

V3 Holdout Accuracy: 0.9858

Classification Report:

              precision    recall  f1-score   support

           0     0.9955    0.9714    0.9833     20492
           1     0.9787    0.9967    0.9876     27022

    accuracy                         0.9858     47514
   macro avg     0.9871    0.9840    0.9855     47514
weighted avg     0.9859    0.9858    0.9858     47514


Confusion Matrix:
[[19905   587]
 [   89 26933]]


In [ ]:
# =========================================================
# V3 - REAL LONG URL TEST
# =========================================================

real_long_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

real_long_features = pd.DataFrame(
    [
        extract_url_features(
            real_long_url
        )
    ]
)[ROBUSTNESS_FEATURES]


prediction = structural_candidate_pipeline_v3.predict(
    real_long_features
)[0]

probabilities = structural_candidate_pipeline_v3.predict_proba(
    real_long_features
)[0]

classes = structural_candidate_pipeline_v3.classes_

probability_map = dict(
    zip(
        classes,
        probabilities,
    )
)


print("URL:")
print(real_long_url)

print("\nPrediction:")
print(
    "Legitimate"
    if prediction == 1
    else "Phishing"
)

print(
    "\nLegitimate probability:",
    f"{probability_map.get(1, 0) * 100:.2f}%"
)

print(
    "Phishing probability:",
    f"{probability_map.get(0, 0) * 100:.2f}%"
)

URL:
https://www.google.com/maps/@19.1922063,72.9234965,11.82z?entry=ttu&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D

Prediction:
Legitimate

Legitimate probability: 85.33%
Phishing probability: 14.67%


In [ ]:
# =========================================================
# V3 - PHISHING STRESS TEST
# =========================================================

phishing_test_urls = [
    "https://paypal-login-security.com",
    "https://account-verification-update.com/login",
    "https://secure-bank-login.xyz/account",
    "https://verify-your-account.com/update",
    "https://login-security-check.net/verify",

    # Query-heavy phishing-style URLs
    "https://paypal-login-security.com/account?id=12345",
    "https://secure-bank-login.xyz/login?session=1234567890",
    "https://verify-your-account.com/update?user=12345&token=abcdef",
    "https://account-verification-update.com/login?redirect=%2Faccount%2Fverify",
    "https://login-security-check.net/verify?id=12345678901234567890&source=web",
]

print("=" * 90)
print("V3 PHISHING STRESS TEST")
print("=" * 90)

for url in phishing_test_urls:

    features = pd.DataFrame(
        [extract_url_features(url)]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline_v3.predict(
        features
    )[0]

    probabilities = structural_candidate_pipeline_v3.predict_proba(
        features
    )[0]

    probability_map = dict(
        zip(
            structural_candidate_pipeline_v3.classes_,
            probabilities,
        )
    )

    phishing_prob = probability_map.get(0, 0) * 100
    legit_prob = probability_map.get(1, 0) * 100

    print("\nURL:", url)

    print(
        "Prediction:",
        "Phishing"
        if prediction == 0
        else "Legitimate"
    )

    print(
        f"Phishing: {phishing_prob:.2f}%"
        f" | Legitimate: {legit_prob:.2f}%"
    )
    

V3 PHISHING STRESS TEST

URL: https://paypal-login-security.com
Prediction: Phishing
Phishing: 97.33% | Legitimate: 2.67%

URL: https://account-verification-update.com/login
Prediction: Phishing
Phishing: 65.00% | Legitimate: 35.00%

URL: https://secure-bank-login.xyz/account
Prediction: Phishing
Phishing: 94.33% | Legitimate: 5.67%

URL: https://verify-your-account.com/update
Prediction: Phishing
Phishing: 79.16% | Legitimate: 20.84%

URL: https://login-security-check.net/verify
Prediction: Phishing
Phishing: 77.60% | Legitimate: 22.40%

URL: https://paypal-login-security.com/account?id=12345
Prediction: Legitimate
Phishing: 34.33% | Legitimate: 65.67%

URL: https://secure-bank-login.xyz/login?session=1234567890
Prediction: Phishing
Phishing: 91.33% | Legitimate: 8.67%

URL: https://verify-your-account.com/update?user=12345&token=abcdef
Prediction: Legitimate
Phishing: 48.67% | Legitimate: 51.33%

URL: https://account-verification-update.com/login?redirect=%2Faccount%2Fverify
Predicti

In [ ]:
# =========================================================
# V4 - GET TRAINING PHISHING URLS
# =========================================================

phishing_train_urls = (
    domain_train_df.loc[
        domain_train_df["label"] == 0,
        "URL"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print(
    "Training phishing URLs:",
    len(phishing_train_urls)
)

print("\nExamples:")

for url in phishing_train_urls[:10]:
    print(url)

Training phishing URLs: 80028

Examples:
http://www.teramill.com
http://www.shprakserf.gq
https://liuy-9a930.web.app/
https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqjgsdn4c3dwkusfdwzdz3kn?clientid=noc@protocol.ai
http://att-103731-107123.weeblysite.com/
http://www.ooguy.com
http://www.fairytalesinc.com
http://www.iuhjn.pplink.club
https://mechinchem-5cb8a.web.app/
https://fb-restriction-case-97be5.web.app/


In [ ]:
# =========================================================
# V4 - GENERATE QUERY-HEAVY PHISHING AUGMENTATION
# =========================================================

import random
from urllib.parse import urlsplit, urlunsplit

random.seed(42)

PHISHING_QUERY_SUFFIXES = [
    "?id=12345",
    "?session=1234567890",
    "?user=12345&token=abcdef",
    "?source=web&lang=en",

    "?redirect=%2Faccount%2Fverify",
    "?next=%2Flogin%2Fverify",

    "?id=12345678901234567890&source=web",

    "?token=ABCDEF12345678901234567890",

    "?data=ExampleLongValue1234567890%3D%3D",

    "?session=ABCDEF12345678901234567890"
    "&redirect=%2Faccount%2Fverify",

    "?lat=19.1922063&lng=72.9234965",

    "?entry=web"
    "&data=AbCdEf12345678901234567890%3D%3D",
]


phishing_query_pool = []

for original_url in phishing_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        # Keep the original phishing domain
        # and preserve its existing path when available
        path = parsed.path

        if not path or path == "/":
            path = "/login"

        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                path,
                "",
                "",
            )
        )

        for suffix in PHISHING_QUERY_SUFFIXES:
            phishing_query_pool.append(
                base_url + suffix
            )

    except Exception:
        continue


print(
    "Phishing query-heavy pool:",
    len(phishing_query_pool)
)


PHISHING_QUERY_SAMPLE_SIZE = 50_000

phishing_query_sample = random.sample(
    phishing_query_pool,
    k=min(
        PHISHING_QUERY_SAMPLE_SIZE,
        len(phishing_query_pool),
    ),
)


print(
    "Phishing query sample:",
    len(phishing_query_sample)
)

print("\nExamples:")

for url in phishing_query_sample[:10]:
    print(url)

Phishing query-heavy pool: 960336
Phishing query sample: 50000

Examples:
http://www.nolanblog.com/login?entry=web&data=AbCdEf12345678901234567890%3D%3D
https://ccxcdff344343dfvccvn.godaddysites.com/login?source=web&lang=en
https://football.abenterprises.com.au/www/mgoqh00jxqnewrd1035qk4kv02ix/dashboad/otp.html?next=%2Flogin%2Fverify
https://fnb0-1491f.firebaseapp.com/login?data=ExampleLongValue1234567890%3D%3D
https://www.attemplate.com/nam/1c98f461-8658-4ef0-a1ed-1ff2b354dad2/888caddd-93af-45b2-80d2-6957c0ff08c8/1f5930cf-6f9a-46b8-ad1e-f1f040d59031/login?next=%2Flogin%2Fverify
https://ionostuedaiy008.firebaseapp.com/login?entry=web&data=AbCdEf12345678901234567890%3D%3D
http://www.kinfdcxv.cf/login?next=%2Flogin%2Fverify
http://www.linkundlink.de/login?id=12345
https://settings40393400390.firebaseapp.com/login?lat=19.1922063&lng=72.9234965
https://id100596210962092061.firebaseapp.com/login?session=1234567890


In [ ]:
# =========================================================
# V4 - EXTRACT PHISHING QUERY FEATURES
# =========================================================

phishing_query_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in phishing_query_sample
    ]
)

phishing_query_features_df.insert(
    0,
    "URL",
    phishing_query_sample
)

phishing_query_features_df["label"] = 0


print(
    "Phishing query augmentation shape:",
    phishing_query_features_df.shape
)

print(
    "Feature count:",
    phishing_query_features_df[
        ROBUSTNESS_FEATURES
    ].shape[1]
)

print(
    "Missing values:",
    phishing_query_features_df[
        ROBUSTNESS_FEATURES
    ].isnull().sum().sum()
)

print("\nLabel counts:")
print(
    phishing_query_features_df[
        "label"
    ].value_counts()
)

print("\nURL Length:")
print(
    phishing_query_features_df[
        "URLLength"
    ].describe()
)

print("\nDigits:")
print(
    phishing_query_features_df[
        "NoOfDegitsInURL"
    ].describe()
)

print("\nObfuscated characters:")
print(
    phishing_query_features_df[
        "NoOfObfuscatedChar"
    ].describe()
)

Phishing query augmentation shape: (50000, 19)
Feature count: 17
Missing values: 0

Label counts:
label
0    50000
Name: count, dtype: int64

URL Length:
count    50000.000000
mean        75.634340
std         44.536859
min         26.000000
25%         59.000000
50%         70.000000
75%         87.000000
max       4309.000000
Name: URLLength, dtype: float64

Digits:
count    50000.000000
mean        14.807540
std         19.825949
min          0.000000
25%          5.000000
50%         14.000000
75%         22.000000
max       2013.000000
Name: NoOfDegitsInURL, dtype: float64

Obfuscated characters:
count    50000.000000
mean         2.552700
std          5.060037
min          0.000000
25%          0.000000
50%          0.000000
75%          6.000000
max        453.000000
Name: NoOfObfuscatedChar, dtype: float64


In [ ]:
# =========================================================
# BUILD V4 FINAL TRAINING DATA
# Original
# + 100k Legitimate Path Augmentation
# + 50k Legitimate Query Augmentation
# + 50k Phishing Query Augmentation
# =========================================================

final_train_df_v4 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
        phishing_query_features_df,
    ],
    ignore_index=True,
)


X_train_final_v4 = final_train_df_v4[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v4 = final_train_df_v4[
    "label"
].copy()


# SAME untouched holdout
X_holdout_v4 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v4 = domain_holdout_df[
    "label"
].copy()


print(
    "V4 training rows:",
    len(final_train_df_v4)
)

print(
    "Holdout rows:",
    len(domain_holdout_df)
)

print(
    "Feature count:",
    X_train_final_v4.shape[1]
)

print(
    "Missing training values:",
    X_train_final_v4.isnull().sum().sum()
)

print("\nV4 training labels:")
print(
    y_train_final_v4.value_counts()
)

V4 training rows: 387856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

V4 training labels:
label
1    257828
0    130028
Name: count, dtype: int64


In [ ]:
# =========================================================
# TRAIN V4 CANDIDATE MODEL
# =========================================================

categorical_features_v4 = ["TLD"]

numeric_features_v4 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]


preprocessor_v4 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v4,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v4,
        ),
    ]
)


candidate_model_v4 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)


structural_candidate_pipeline_v4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v4),
    ]
)


print("Training V4 candidate...")

structural_candidate_pipeline_v4.fit(
    X_train_final_v4,
    y_train_final_v4,
)

print("V4 candidate training complete.")

Training V4 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V4 candidate training complete.


In [87]:
# =========================================================
# V4 - COMBINED EVALUATION
# Holdout + Legit Paths + Real Long URL + Phishing Stress
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

print("=" * 90)
print("1) UNTOUCHED HOLDOUT")
print("=" * 90)

y_pred_v4 = structural_candidate_pipeline_v4.predict(
    X_holdout_v4
)

accuracy_v4 = accuracy_score(
    y_holdout_v4,
    y_pred_v4,
)

print(
    "V4 Holdout Accuracy:",
    round(accuracy_v4, 4)
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_holdout_v4,
        y_pred_v4,
        digits=4,
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_holdout_v4,
        y_pred_v4,
    )
)


# =========================================================
# HELPER FUNCTION
# =========================================================

def test_v4_url(url):

    features = pd.DataFrame(
        [extract_url_features(url)]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline_v4.predict(
        features
    )[0]

    probabilities = structural_candidate_pipeline_v4.predict_proba(
        features
    )[0]

    probability_map = dict(
        zip(
            structural_candidate_pipeline_v4.classes_,
            probabilities,
        )
    )

    phishing_prob = probability_map.get(0, 0) * 100
    legit_prob = probability_map.get(1, 0) * 100

    label = (
        "Legitimate"
        if prediction == 1
        else "Phishing"
    )

    print("\nURL:", url)
    print("Prediction:", label)
    print(
        f"Legitimate: {legit_prob:.2f}%"
        f" | Phishing: {phishing_prob:.2f}%"
    )


# =========================================================
# 2) OLD LEGITIMATE PATH TESTS
# =========================================================

print("\n" + "=" * 90)
print("2) LEGITIMATE PATH TESTS")
print("=" * 90)

legitimate_test_urls = [
    "https://www.google.com",
    "https://www.google.com/maps",
    "https://www.google.com/maps/place",
    "https://www.google.com/a/b",
    "https://www.google.com/a/b/c",
    "https://www.paypal.com",
    "https://www.paypal.com/home",
    "https://www.paypal.com/in/home",
]

for url in legitimate_test_urls:
    test_v4_url(url)


# =========================================================
# 3) REAL LONG LEGITIMATE URL
# =========================================================

print("\n" + "=" * 90)
print("3) REAL LONG LEGITIMATE URL")
print("=" * 90)

real_long_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

test_v4_url(
    real_long_url
)


# =========================================================
# 4) PHISHING STRESS TESTS
# =========================================================

print("\n" + "=" * 90)
print("4) PHISHING STRESS TESTS")
print("=" * 90)

phishing_test_urls = [
    "https://paypal-login-security.com",
    "https://account-verification-update.com/login",
    "https://secure-bank-login.xyz/account",
    "https://verify-your-account.com/update",
    "https://login-security-check.net/verify",

    "https://paypal-login-security.com/account?id=12345",

    "https://secure-bank-login.xyz/login?session=1234567890",

    "https://verify-your-account.com/update?user=12345&token=abcdef",

    "https://account-verification-update.com/login?redirect=%2Faccount%2Fverify",

    "https://login-security-check.net/verify?id=12345678901234567890&source=web",
]

for url in phishing_test_urls:
    test_v4_url(url)

1) UNTOUCHED HOLDOUT
V4 Holdout Accuracy: 0.9834

Classification Report:

              precision    recall  f1-score   support

           0     0.9966    0.9648    0.9805     20492
           1     0.9739    0.9975    0.9856     27022

    accuracy                         0.9834     47514
   macro avg     0.9853    0.9812    0.9830     47514
weighted avg     0.9837    0.9834    0.9834     47514

Confusion Matrix:
[[19771   721]
 [   67 26955]]

2) LEGITIMATE PATH TESTS

URL: https://www.google.com
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/place
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/a/b
Prediction: Legitimate
Legitimate: 78.33% | Phishing: 21.67%

URL: https://www.google.com/a/b/c
Prediction: Legitimate
Legitimate: 68.00% | Phishing: 32.00%

URL: https://www.paypal.com
Prediction: Legitim

In [ ]:
long_legitimate_tests = [
    "https://example.com/search?q=hello%20world&page=2",
    "https://example.com/location?lat=19.1922063&lng=72.9234965",
    "https://example.com/docs/view?id=12345&lang=en",

    "https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view",

    "https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&data=ExampleLongValue1234567890%3D%3D",

    "https://example.com/docs/view?id=12345678901234567890&source=browser&lang=en",

    "https://example.com/redirect?next=%2Faccount%2Fsettings%2Fprivacy&source=web",

    "https://example.com/place/@51.507351,-0.127758,14.25z?entry=browser&data=Example1234567890%3D%3D",
]

print("LONG LEGITIMATE TESTS")

for url in long_legitimate_tests:
    test_v4_url(url)

LONG LEGITIMATE TESTS

URL: https://example.com/search?q=hello%20world&page=2
Prediction: Legitimate
Legitimate: 85.33% | Phishing: 14.67%

URL: https://example.com/location?lat=19.1922063&lng=72.9234965
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/docs/view?id=12345&lang=en
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&data=ExampleLongValue1234567890%3D%3D
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/docs/view?id=12345678901234567890&source=browser&lang=en
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/redirect?next=%2Faccount%2Fsettings%2Fprivacy&source=web
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https:/

In [88]:
# =========================================================
# SAVE FINAL V4 PRODUCTION MODEL
# =========================================================

import joblib
from pathlib import Path

production_model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

joblib.dump(
    structural_candidate_pipeline_v4,
    production_model_path
)

print(
    "Saved V4 production model to:",
    production_model_path
)

Saved V4 production model to: ..\models\trustlens_production_pipeline.joblib


In [ ]:
# =========================================================
# SAVE FINAL V4 PRODUCTION MODEL
# =========================================================

import joblib
from pathlib import Path

production_model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

joblib.dump(
    structural_candidate_pipeline_v4,
    production_model_path
)

print("Saved V4 production model to:", production_model_path)

NameError: name 'structural_candidate_pipeline_v4' is not defined

In [ ]:
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

print("ROBUSTNESS_FEATURES restored:", len(ROBUSTNESS_FEATURES))

ROBUSTNESS_FEATURES restored: 17


In [28]:
# =========================================================
# RECOVERY STEP 2 - REBUILD V4 TRAIN / HOLDOUT MATRICES
# =========================================================

final_train_df_v4 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
        phishing_query_features_df,
        
    ],
    ignore_index=True,
)

X_train_final_v4 = final_train_df_v4[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v4 = final_train_df_v4[
    "label"
].copy()

X_holdout_v4 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v4 = domain_holdout_df[
    "label"
].copy()

print("V4 training rows:", len(final_train_df_v4))
print("Holdout rows:", len(domain_holdout_df))
print("Feature count:", X_train_final_v4.shape[1])
print("Missing values:", X_train_final_v4.isnull().sum().sum())

print("\nV4 label counts:")
print(y_train_final_v4.value_counts())

V4 training rows: 387856
Holdout rows: 47514
Feature count: 17
Missing values: 0

V4 label counts:
label
1    257828
0    130028
Name: count, dtype: int64


In [ ]:
required_vars = [
    "domain_train_df",
    "domain_holdout_df",
    "augmented_features_df",
    "better_query_features_df",
    "phishing_query_features_df",
    "extract_url_features",
]

for name in required_vars:
    print(f"{name}: {'EXISTS' if name in globals() else 'MISSING'}")

domain_train_df: EXISTS
domain_holdout_df: EXISTS
augmented_features_df: EXISTS
better_query_features_df: MISSING
phishing_query_features_df: MISSING
extract_url_features: EXISTS


In [27]:
# =========================================================
# RECOVERY STEP 3
# Recreate V3 legitimate query augmentation
# + V4 phishing query augmentation
# =========================================================

import random
import pandas as pd
from urllib.parse import urlparse, urlunparse

# ---------------------------------------------------------
# A) RECREATE BETTER LEGITIMATE QUERY AUGMENTATION (V3)
# ---------------------------------------------------------

legitimate_train_urls = (
    domain_train_df[
        domain_train_df["label"] == 1
    ]["URL"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

NORMAL_QUERY_TEMPLATES = [
    "/search?q=example",
    "/search?q=example&page=2",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.9234965",
    "/search?q=hello%20world",
    "/redirect?next=%2Faccount%2Fsettings",
]

COMPLEX_QUERY_TEMPLATES = [
    "/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view",
    "/location/@19.1922063,72.9234965,11.82z?entry=web&source=browser&mode=view",
    "/location/@19.1922063,72.9234965,11.82z?entry=web&data=ExampleLongValue1234567890%3D%3D",
    "/location/@19.1922063,72.9234965,11.82z?source=browser&data=AbCdEf12345678901234567890%3D%3D",
    "/view/item/1234567890?session=ABCDEF12345678901234567890&lang=en",
    "/docs/view?id=12345678901234567890&source=browser&lang=en",
    "/redirect?next=%2Faccount%2Fsettings%2Fprivacy&source=web",
    "/search?q=hello%20world&category=articles&page=123456",
    "/data/@40.712776,-74.005974,12.50z?view=detail&source=web",
    "/place/@51.507351,-0.127758,14.25z?entry=browser&data=Example1234567890%3D%3D",
]

normal_query_pool = []
complex_query_pool = []

for url in legitimate_train_urls:
    parsed = urlparse(url)

    scheme = parsed.scheme if parsed.scheme else "https"
    netloc = parsed.netloc

    if not netloc:
        continue

    for template in NORMAL_QUERY_TEMPLATES:
        normal_query_pool.append(
            f"{scheme}://{netloc}{template}"
        )

    for template in COMPLEX_QUERY_TEMPLATES:
        complex_query_pool.append(
            f"{scheme}://{netloc}{template}"
        )

random.seed(42)

sampled_normal_urls = random.sample(
    normal_query_pool,
    25000
)

sampled_complex_urls = random.sample(
    complex_query_pool,
    25000
)

better_query_urls = (
    sampled_normal_urls
    + sampled_complex_urls
)

better_query_features_df = pd.DataFrame(
    [
        {
            **extract_url_features(url),
            "URL": url,
            "label": 1,
        }
        for url in better_query_urls
    ]
)


# ---------------------------------------------------------
# B) RECREATE PHISHING QUERY AUGMENTATION (V4)
# ---------------------------------------------------------

phishing_train_urls = (
    domain_train_df[
        domain_train_df["label"] == 0
    ]["URL"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

PHISHING_QUERY_SUFFIXES = [
    "?id=12345",
    "?session=1234567890",
    "?user=12345&token=abcdef",
    "?source=web&lang=en",
    "?redirect=%2Faccount%2Fverify",
    "?next=%2Flogin%2Fverify",
    "?id=12345678901234567890&source=web",
    "?token=ABCDEF12345678901234567890",
    "?data=ExampleLongValue1234567890%3D%3D",
    "?session=ABCDEF12345678901234567890&redirect=%2Faccount%2Fverify",
    "?lat=19.1922063&lng=72.9234965",
    "?entry=web&data=AbCdEf12345678901234567890%3D%3D",
]

phishing_query_pool = []

for url in phishing_train_urls:
    parsed = urlparse(url)

    scheme = parsed.scheme if parsed.scheme else "https"
    netloc = parsed.netloc

    if not netloc:
        continue

    path = parsed.path

    if not path or path == "/":
        path = "/login"

    base_url = urlunparse(
        (
            scheme,
            netloc,
            path,
            "",
            "",
            "",
        )
    )

    for suffix in PHISHING_QUERY_SUFFIXES:
        phishing_query_pool.append(
            base_url + suffix
        )

random.seed(42)

sampled_phishing_query_urls = random.sample(
    phishing_query_pool,
    50000
)

phishing_query_features_df = pd.DataFrame(
    [
        {
            **extract_url_features(url),
            "URL": url,
            "label": 0,
        }
        for url in sampled_phishing_query_urls
    ]
)


# ---------------------------------------------------------
# VERIFY
# ---------------------------------------------------------

print("better_query_features_df:", better_query_features_df.shape)
print("phishing_query_features_df:", phishing_query_features_df.shape)

print("\nLegitimate query labels:")
print(better_query_features_df["label"].value_counts())

print("\nPhishing query labels:")
print(phishing_query_features_df["label"].value_counts())

better_query_features_df: (50000, 19)
phishing_query_features_df: (50000, 19)

Legitimate query labels:
label
1    50000
Name: count, dtype: int64

Phishing query labels:
label
0    50000
Name: count, dtype: int64


In [ ]:
# =========================================================
# RECOVERY STEP 4 - REBUILD V4 TRAIN / HOLDOUT MATRICES
# =========================================================

final_train_df_v4 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
        phishing_query_features_df,
    ],
    ignore_index=True,
)

X_train_final_v4 = final_train_df_v4[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v4 = final_train_df_v4[
    "label"
].copy()

X_holdout_v4 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v4 = domain_holdout_df[
    "label"
].copy()

print("V4 training rows:", len(final_train_df_v4))
print("Holdout rows:", len(domain_holdout_df))
print("Feature count:", X_train_final_v4.shape[1])
print("Missing values:", X_train_final_v4.isnull().sum().sum())

print("\nV4 label counts:")
print(y_train_final_v4.value_counts())

V4 training rows: 387856
Holdout rows: 47514
Feature count: 17
Missing values: 0

V4 label counts:
label
1    257828
0    130028
Name: count, dtype: int64


In [85]:
# =========================================================
# RECOVERY STEP 5 - DEFINE + TRAIN V4 CANDIDATE
# =========================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

categorical_features_v4 = ["TLD"]

numeric_features_v4 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]

preprocessor_v4 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v4,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v4,
        ),
    ]
)

candidate_model_v4 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v4),
    ]
)

print("Training V4 candidate...")

structural_candidate_pipeline_v4.fit(
    X_train_final_v4,
    y_train_final_v4,
)

print("V4 candidate training complete.")

Training V4 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V4 candidate training complete.


In [86]:
def test_v4_candidate_url(url):
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v4.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v4.predict_proba(input_df)[0]

    classes = list(structural_candidate_pipeline_v4.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    prediction_label = "Phishing" if prediction == 0 else "Legitimate"

    print(f"\nURL: {url}")
    print(f"Prediction: {prediction_label}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )


v4_candidate_test_urls = [
    "https://www.google.com",

    "https://www.google.com/search?q=hello%20world&page=2",

    "https://www.google.com/location?lat=19.1922063&lng=72.8777",

    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",

    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",

    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser",

    "http://secure-login-example.com/verify/account",

    "https://account-verification-update.com/login",
]

for url in v4_candidate_test_urls:
    test_v4_candidate_url(url)


URL: https://www.google.com
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/search?q=hello%20world&page=2
Prediction: Legitimate
Legitimate: 86.67% | Phishing: 13.33%

URL: https://www.google.com/location?lat=19.1922063&lng=72.8777
Prediction: Legitimate
Legitimate: 69.33% | Phishing: 30.67%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 1.33% | Phishing: 98.67%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
Prediction: Phishing
Legitimate: 4.33% | Phishing: 95.67%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser
Prediction: Legitimate
Legitimate: 57.67% | Phishing: 42.33%

URL: http://secure-login-example.com/verify/account
Prediction: Phishing
Legitimate: 6.67% | Phishing: 93.33%

URL: https://account-verification-update.com/login
Prediction: Phishing
Legitimate: 30.67% | Phishing: 69.33%


In [31]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

v4_holdout_pred = structural_candidate_pipeline_v4.predict(X_holdout_v4)

print("V4 Candidate Holdout Accuracy:")
print(accuracy_score(y_holdout_v4, v4_holdout_pred))

print("\nV4 Candidate Classification Report:")
print(
    classification_report(
        y_holdout_v4,
        v4_holdout_pred,
        target_names=["Phishing", "Legitimate"]
    )
)

print("\nV4 Candidate Confusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v4,
        v4_holdout_pred
    )
)

V4 Candidate Holdout Accuracy:
0.9834154144041756

V4 Candidate Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      0.96      0.98     20492
  Legitimate       0.97      1.00      0.99     27022

    accuracy                           0.98     47514
   macro avg       0.99      0.98      0.98     47514
weighted avg       0.98      0.98      0.98     47514


V4 Candidate Confusion Matrix:
[[19771   721]
 [   67 26955]]


In [32]:
# V4 CANDIDATE FEATURE IMPORTANCE

rf_v4 = structural_candidate_pipeline_v4.named_steps["model"]

feature_names_v4 = (
    numeric_features_v4
    + ["TLD_encoded"]
)

importances_v4 = rf_v4.feature_importances_

feature_importance_v4 = sorted(
    zip(feature_names_v4, importances_v4),
    key=lambda x: x[1],
    reverse=True
)

print("V4 Candidate Feature Importance")
print("=" * 50)

for feature, importance in feature_importance_v4:
    print(f"{feature:25s} {importance:.4f}")

V4 Candidate Feature Importance
NoOfDots                  0.3071
URLLength                 0.1588
NoOfEqualsInURL           0.1347
IsDomainIP                0.0660
HasHyphenInDomain         0.0532
SuspiciousKeywordCount    0.0502
DomainLength              0.0469
TLD_encoded               0.0411
PathDepth                 0.0296
HasObfuscation            0.0285
NoOfSlashes               0.0283
NoOfQMarkInURL            0.0262
IsHTTPS                   0.0131
NoOfDegitsInURL           0.0081
NoOfSubDomain             0.0056
NoOfObfuscatedChar        0.0023
TLDLength                 0.0000


In [33]:
# V4 CANDIDATE - GOOGLE MAPS FEATURE PERTURBATION

maps_url = "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu"

maps_features = extract_url_features(maps_url)
maps_df = pd.DataFrame([maps_features])

classes = list(structural_candidate_pipeline_v4.classes_)

baseline_probs = structural_candidate_pipeline_v4.predict_proba(maps_df)[0]

print("BASELINE GOOGLE MAPS")
print(
    f"Legitimate: {baseline_probs[classes.index(1)]:.2%} | "
    f"Phishing: {baseline_probs[classes.index(0)]:.2%}"
)

features_to_test = [
    "URLLength",
    "NoOfDots",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfSlashes",
    "NoOfDegitsInURL",
    "PathDepth",
]

print("\nFEATURE PERTURBATION")
print("=" * 60)

for feature in features_to_test:

    modified_features = maps_features.copy()

    # Replace the feature with a simpler value
    if feature in ["NoOfDots", "NoOfEqualsInURL", "NoOfQMarkInURL",
                   "NoOfSlashes", "NoOfDegitsInURL", "PathDepth"]:
        modified_features[feature] = 0
    elif feature == "URLLength":
        modified_features[feature] = 30

    modified_df = pd.DataFrame([modified_features])

    probs = structural_candidate_pipeline_v4.predict_proba(modified_df)[0]

    legitimate = probs[classes.index(1)]
    phishing = probs[classes.index(0)]

    print(
        f"{feature:20s} -> "
        f"Legitimate: {legitimate:.2%} | "
        f"Phishing: {phishing:.2%}"
    )

BASELINE GOOGLE MAPS
Legitimate: 1.33% | Phishing: 98.67%

FEATURE PERTURBATION
URLLength            -> Legitimate: 7.00% | Phishing: 93.00%
NoOfDots             -> Legitimate: 1.00% | Phishing: 99.00%
NoOfEqualsInURL      -> Legitimate: 1.00% | Phishing: 99.00%
NoOfQMarkInURL       -> Legitimate: 1.00% | Phishing: 99.00%
NoOfSlashes          -> Legitimate: 12.67% | Phishing: 87.33%
NoOfDegitsInURL      -> Legitimate: 1.00% | Phishing: 99.00%
PathDepth            -> Legitimate: 3.67% | Phishing: 96.33%


In [34]:
comparison_urls = [
    "https://www.google.com",
    "https://www.google.com/search?q=hello%20world&page=2",
    "https://www.google.com/location?lat=19.1922063&lng=72.8777",
    "https://www.google.com/maps",
    "https://www.google.com/maps/",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
]

print("GOOGLE STRUCTURAL COMPARISON")
print("=" * 70)

for url in comparison_urls:
    test_v4_candidate_url(url)

GOOGLE STRUCTURAL COMPARISON

URL: https://www.google.com
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/search?q=hello%20world&page=2
Prediction: Legitimate
Legitimate: 86.67% | Phishing: 13.33%

URL: https://www.google.com/location?lat=19.1922063&lng=72.8777
Prediction: Legitimate
Legitimate: 69.33% | Phishing: 30.67%

URL: https://www.google.com/maps
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z
Prediction: Phishing
Legitimate: 0.33% | Phishing: 99.67%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 1.33% | Phishing: 98.67%


In [35]:
maps_structure_tests = [
    "https://www.google.com/maps",
    "https://www.google.com/maps/",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z",
]

for url in maps_structure_tests:
    features = extract_url_features(url)

    print("\n" + "=" * 70)
    print(url)

    for feature in ROBUSTNESS_FEATURES:
        print(f"{feature:25s}: {features[feature]}")


https://www.google.com/maps
URLLength                : 27
DomainLength             : 14
IsDomainIP               : 0
TLD                      : com
TLDLength                : 3
NoOfSubDomain            : 1
HasObfuscation           : 0
NoOfObfuscatedChar       : 0
NoOfDegitsInURL          : 0
NoOfEqualsInURL          : 0
NoOfQMarkInURL           : 0
IsHTTPS                  : 1
NoOfDots                 : 2
NoOfSlashes              : 3
SuspiciousKeywordCount   : 0
HasHyphenInDomain        : 0
PathDepth                : 1

https://www.google.com/maps/
URLLength                : 28
DomainLength             : 14
IsDomainIP               : 0
TLD                      : com
TLDLength                : 3
NoOfSubDomain            : 1
HasObfuscation           : 0
NoOfObfuscatedChar       : 0
NoOfDegitsInURL          : 0
NoOfEqualsInURL          : 0
NoOfQMarkInURL           : 0
IsHTTPS                  : 1
NoOfDots                 : 2
NoOfSlashes              : 4
SuspiciousKeywordCount   : 0
HasHy

In [36]:
maps_slash_test = "https://www.google.com/maps/"

base_features = extract_url_features(maps_slash_test)
base_df = pd.DataFrame([base_features])

print("BASELINE")
test_v4_candidate_url(maps_slash_test)

for feature_name, new_value in [
    ("NoOfSlashes", 3),
    ("PathDepth", 0),
]:
    modified_features = base_features.copy()
    modified_features[feature_name] = new_value

    modified_df = pd.DataFrame([modified_features])

    prediction = structural_candidate_pipeline_v4.predict(modified_df)[0]
    probabilities = structural_candidate_pipeline_v4.predict_proba(modified_df)[0]
    classes = list(structural_candidate_pipeline_v4.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    print("\n" + "-" * 60)
    print(f"Changed {feature_name} -> {new_value}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )

BASELINE

URL: https://www.google.com/maps/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

------------------------------------------------------------
Changed NoOfSlashes -> 3
Legitimate: 100.00% | Phishing: 0.00%

------------------------------------------------------------
Changed PathDepth -> 0
Legitimate: 26.41% | Phishing: 73.59%


In [37]:
slash_generalization_urls = [
    "https://www.google.com/maps/",
    "https://www.google.com/about/",
    "https://www.google.com/contact/",
    "https://www.google.com/help/",
    "https://www.google.com/products/",
    "https://www.google.com/docs/",
]

print("4-SLASH LEGITIMATE URL TEST")
print("=" * 70)

for url in slash_generalization_urls:
    test_v4_candidate_url(url)

4-SLASH LEGITIMATE URL TEST

URL: https://www.google.com/maps/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.google.com/about/
Prediction: Phishing
Legitimate: 8.67% | Phishing: 91.33%

URL: https://www.google.com/contact/
Prediction: Phishing
Legitimate: 8.33% | Phishing: 91.67%

URL: https://www.google.com/help/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.google.com/products/
Prediction: Phishing
Legitimate: 10.67% | Phishing: 89.33%

URL: https://www.google.com/docs/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%


In [38]:
print("NoOfSlashes distribution in V4 training data")
print("=" * 70)

print("\nLEGITIMATE:")
print(
    final_train_df_v4[
        final_train_df_v4["label"] == 1
    ]["NoOfSlashes"].value_counts().sort_index()
)

print("\nPHISHING:")
print(
    final_train_df_v4[
        final_train_df_v4["label"] == 0
    ]["NoOfSlashes"].value_counts().sort_index()
)

NoOfSlashes distribution in V4 training data

LEGITIMATE:
NoOfSlashes
2    107828
3     62312
4     51696
5     23452
6     12540
Name: count, dtype: int64

PHISHING:
NoOfSlashes
2     32480
3     72568
4     12936
5      6365
6      2364
7      1292
8       858
9       419
10      241
11      324
12       93
13       22
14       23
15        5
16        6
18        4
19        1
20        2
21        1
25        1
26        3
29        5
30        5
31        1
32        3
33        1
34        2
35        2
68        1
Name: count, dtype: int64


In [39]:
legit_4_slash = final_train_df_v4[
    (final_train_df_v4["label"] == 1) &
    (final_train_df_v4["NoOfSlashes"] == 4)
]

print("Legitimate URLs with exactly 4 slashes:", len(legit_4_slash))
print("\nFeature ranges:")
print(
    legit_4_slash[
        [
            "URLLength",
            "NoOfDots",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfSlashes",
            "PathDepth",
        ]
    ].describe()
)

Legitimate URLs with exactly 4 slashes: 51696

Feature ranges:
          URLLength      NoOfDots  NoOfDegitsInURL  NoOfEqualsInURL  \
count  51696.000000  51696.000000     51696.000000     51696.000000   
mean      61.201776      2.757834         9.646472         0.922025   
std       29.170314      1.646254        13.817894         1.112605   
min       24.000000      1.000000         0.000000         0.000000   
25%       38.000000      2.000000         0.000000         0.000000   
50%       46.000000      2.000000         0.000000         0.000000   
75%       86.000000      5.000000        20.000000         2.000000   
max      143.000000      8.000000        52.000000         3.000000   

       NoOfQMarkInURL  NoOfSlashes  PathDepth  
count    51696.000000      51696.0    51696.0  
mean         0.436339          4.0        2.0  
std          0.495936          0.0        0.0  
min          0.000000          4.0        2.0  
25%          0.000000          4.0        2.0  
50%      

In [40]:
phish_4_slash = final_train_df_v4[
    (final_train_df_v4["label"] == 0) &
    (final_train_df_v4["NoOfSlashes"] == 4)
]

print("Phishing URLs with exactly 4 slashes:", len(phish_4_slash))
print("\nFeature ranges:")
print(
    phish_4_slash[
        [
            "URLLength",
            "NoOfDots",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfSlashes",
            "PathDepth",
        ]
    ].describe()
)

Phishing URLs with exactly 4 slashes: 12936

Feature ranges:
          URLLength      NoOfDots  NoOfDegitsInURL  NoOfEqualsInURL  \
count  12936.000000  12936.000000     12936.000000     12936.000000   
mean      78.500850      2.544991        11.385668         0.773036   
std       78.053436      1.377376        17.200761         1.277670   
min       19.000000      1.000000         0.000000         0.000000   
25%       53.000000      2.000000         2.000000         0.000000   
50%       71.000000      2.000000         8.000000         0.000000   
75%       93.000000      3.000000        18.000000         1.000000   
max     5795.000000     21.000000       763.000000        31.000000   

       NoOfQMarkInURL  NoOfSlashes     PathDepth  
count    12936.000000      12936.0  12936.000000  
mean         0.488095          4.0      1.678571  
std          0.506179          0.0      0.479781  
min          0.000000          4.0      0.000000  
25%          0.000000          4.0      1.00

In [41]:
targeted_legitimate_urls = [
    "https://www.google.com/maps/",
    "https://www.google.com/about/",
    "https://www.google.com/contact/",
    "https://www.google.com/help/",
    "https://www.google.com/docs/",
    "https://www.google.com/products/",
    "https://www.microsoft.com/about/",
    "https://www.microsoft.com/support/",
    "https://www.amazon.com/help/",
    "https://www.wikipedia.org/about/",
]

targeted_phishing_urls = [
    "http://secure-login-example.com/verify/",
    "http://secure-account-update-example.com/login/",
    "http://paypal-login-security-example.com/verify/",
    "https://account-verification-update.com/login/",
]

print("TARGETED LEGITIMATE URLS")
print("=" * 70)

for url in targeted_legitimate_urls:
    test_v4_candidate_url(url)

print("\n\nTARGETED PHISHING URLS")
print("=" * 70)

for url in targeted_phishing_urls:
    test_v4_candidate_url(url)

TARGETED LEGITIMATE URLS

URL: https://www.google.com/maps/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.google.com/about/
Prediction: Phishing
Legitimate: 8.67% | Phishing: 91.33%

URL: https://www.google.com/contact/
Prediction: Phishing
Legitimate: 8.33% | Phishing: 91.67%

URL: https://www.google.com/help/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.google.com/docs/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.google.com/products/
Prediction: Phishing
Legitimate: 10.67% | Phishing: 89.33%

URL: https://www.microsoft.com/about/
Prediction: Phishing
Legitimate: 6.33% | Phishing: 93.67%

URL: https://www.microsoft.com/support/
Prediction: Phishing
Legitimate: 9.67% | Phishing: 90.33%

URL: https://www.amazon.com/help/
Prediction: Phishing
Legitimate: 12.00% | Phishing: 88.00%

URL: https://www.wikipedia.org/about/
Prediction: Phishing
Legitimate: 28.00% | Phishing: 72.00%


TARGETED PHISHI

In [43]:
targeted_trailing_slash_urls = [
    "https://www.google.com/maps/",
    "https://www.google.com/about/",
    "https://www.google.com/contact/",
    "https://www.google.com/help/",
    "https://www.google.com/docs/",
    "https://www.google.com/products/",
    "https://www.microsoft.com/about/",
    "https://www.microsoft.com/support/",
    "https://www.amazon.com/help/",
    "https://www.wikipedia.org/about/",
    "https://www.apple.com/support/",
    "https://www.github.com/about/",
    "https://www.cloudflare.com/about/",
    "https://www.mozilla.org/about/",
    "https://www.adobe.com/support/",
]

targeted_features = pd.DataFrame(
    [extract_url_features(url) for url in targeted_trailing_slash_urls]
)

targeted_features["label"] = 1

print("Targeted legitimate augmentation:")
print(targeted_features.shape)

print("\nFeature distribution:")
print(
    targeted_features[
        [
            "URLLength",
            "NoOfDots",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfSlashes",
            "PathDepth",
        ]
    ].describe()
)

print("\nLabel counts:")
print(targeted_features["label"].value_counts())

Targeted legitimate augmentation:
(15, 18)

Feature distribution:
       URLLength  NoOfDots  NoOfDegitsInURL  NoOfEqualsInURL  NoOfQMarkInURL  \
count  15.000000      15.0             15.0             15.0            15.0   
mean   30.266667       2.0              0.0              0.0             0.0   
std     1.980861       0.0              0.0              0.0             0.0   
min    28.000000       2.0              0.0              0.0             0.0   
25%    28.500000       2.0              0.0              0.0             0.0   
50%    30.000000       2.0              0.0              0.0             0.0   
75%    32.000000       2.0              0.0              0.0             0.0   
max    34.000000       2.0              0.0              0.0             0.0   

       NoOfSlashes  PathDepth  
count         15.0       15.0  
mean           4.0        1.0  
std            0.0        0.0  
min            4.0        1.0  
25%            4.0        1.0  
50%            4.0   

In [44]:
import random

random.seed(42)

legitimate_domains = [
    "google.com",
    "microsoft.com",
    "amazon.com",
    "wikipedia.org",
    "apple.com",
    "github.com",
    "cloudflare.com",
    "mozilla.org",
    "adobe.com",
    "python.org",
    "stackoverflow.com",
    "reddit.com",
    "linkedin.com",
    "dropbox.com",
    "wordpress.com",
]

simple_paths = [
    "about",
    "contact",
    "help",
    "support",
    "docs",
    "products",
    "services",
    "company",
    "privacy",
    "terms",
    "news",
    "blog",
    "resources",
    "features",
    "community",
]

targeted_augmented_urls = []

for domain in legitimate_domains:
    for path in simple_paths:
        targeted_augmented_urls.append(
            f"https://www.{domain}/{path}/"
        )

targeted_augmented_urls = random.sample(
    targeted_augmented_urls,
    200
)

targeted_augmented_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in targeted_augmented_urls
    ]
)

targeted_augmented_features["label"] = 1

print("Targeted augmentation shape:")
print(targeted_augmented_features.shape)

print("\nKey feature distribution:")
print(
    targeted_augmented_features[
        [
            "URLLength",
            "NoOfDots",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfSlashes",
            "PathDepth",
        ]
    ].describe()
)

print("\nLabel counts:")
print(targeted_augmented_features["label"].value_counts())

Targeted augmentation shape:
(200, 18)

Key feature distribution:
        URLLength  NoOfDots  NoOfDegitsInURL  NoOfEqualsInURL  NoOfQMarkInURL  \
count  200.000000     200.0            200.0            200.0           200.0   
mean    31.860000       2.0              0.0              0.0             0.0   
std      2.805307       0.0              0.0              0.0             0.0   
min     27.000000       2.0              0.0              0.0             0.0   
25%     30.000000       2.0              0.0              0.0             0.0   
50%     32.000000       2.0              0.0              0.0             0.0   
75%     34.000000       2.0              0.0              0.0             0.0   
max     40.000000       2.0              0.0              0.0             0.0   

       NoOfSlashes  PathDepth  
count        200.0      200.0  
mean           4.0        1.0  
std            0.0        0.0  
min            4.0        1.0  
25%            4.0        1.0  
50%         

In [45]:
final_train_df_v5 = pd.concat(
    [
        final_train_df_v4,
        targeted_augmented_features,
    ],
    ignore_index=True,
)

X_train_final_v5 = final_train_df_v5[ROBUSTNESS_FEATURES].copy()
y_train_final_v5 = final_train_df_v5["label"].copy()

print("V5 training shape:", final_train_df_v5.shape)
print("\nLabel counts:")
print(y_train_final_v5.value_counts())

candidate_model_v5 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v5 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v5),
    ]
)

print("\nTraining V5 candidate...")

structural_candidate_pipeline_v5.fit(
    X_train_final_v5,
    y_train_final_v5,
)

print("V5 candidate training complete.")

V5 training shape: (388056, 20)

Label counts:
label
1    258028
0    130028
Name: count, dtype: int64

Training V5 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V5 candidate training complete.


In [46]:
def test_v5_candidate_url(url):
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v5.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v5.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v5.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    prediction_label = "Phishing" if prediction == 0 else "Legitimate"

    print(f"\nURL: {url}")
    print(f"Prediction: {prediction_label}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )


print("V5 TARGETED LEGITIMATE TEST")
print("=" * 70)

for url in targeted_legitimate_urls:
    test_v5_candidate_url(url)


print("\n\nV5 TARGETED PHISHING TEST")
print("=" * 70)

for url in targeted_phishing_urls:
    test_v5_candidate_url(url)

V5 TARGETED LEGITIMATE TEST

URL: https://www.google.com/maps/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/about/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/contact/
Prediction: Legitimate
Legitimate: 99.67% | Phishing: 0.33%

URL: https://www.google.com/help/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/docs/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/products/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.microsoft.com/about/
Prediction: Legitimate
Legitimate: 96.33% | Phishing: 3.67%

URL: https://www.microsoft.com/support/
Prediction: Legitimate
Legitimate: 98.33% | Phishing: 1.67%

URL: https://www.amazon.com/help/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.wikipedia.org/about/
Prediction: Legitimate
Legitimate: 99.33% | Phishing: 

In [47]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

v5_holdout_predictions = structural_candidate_pipeline_v5.predict(
    X_holdout_v4
)

v5_holdout_accuracy = accuracy_score(
    y_holdout_v4,
    v5_holdout_predictions
)

print("V5 Candidate Holdout Accuracy:")
print(v5_holdout_accuracy)

print("\nV5 Candidate Classification Report:")
print(
    classification_report(
        y_holdout_v4,
        v5_holdout_predictions,
        target_names=["Phishing", "Legitimate"],
    )
)

print("V5 Candidate Confusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v4,
        v5_holdout_predictions,
    )
)

V5 Candidate Holdout Accuracy:
0.9853727322473377

V5 Candidate Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      0.97      0.98     20492
  Legitimate       0.98      1.00      0.99     27022

    accuracy                           0.99     47514
   macro avg       0.99      0.98      0.99     47514
weighted avg       0.99      0.99      0.99     47514

V5 Candidate Confusion Matrix:
[[19851   641]
 [   54 26968]]


In [48]:
v5_generalization_urls = [
    # Unseen legitimate domains
    "https://www.nasa.gov/about/",
    "https://www.ibm.com/support/",
    "https://www.cisco.com/products/",
    "https://www.intel.com/docs/",
    "https://www.nytimes.com/contact/",
    
    # Non-www legitimate URLs
    "https://google.com/maps/",
    "https://microsoft.com/about/",
    "https://github.com/about/",
    
    # Slightly deeper legitimate paths
    "https://www.google.com/maps/place/",
    "https://www.microsoft.com/en-us/support/",
    "https://www.amazon.com/gp/help/",
    
    # Legitimate query-heavy URLs
    "https://www.google.com/search?q=artificial+intelligence&page=2",
    "https://www.amazon.com/s?k=cybersecurity+books",
    "https://github.com/search?q=phishing&type=repositories",
    
    # Authentication-related legitimate URLs
    "https://login.microsoftonline.com/",
    "https://accounts.google.com/",
    "https://www.amazon.com/ap/signin",
    
    # Real-world Google Maps structure
    "https://www.google.com/maps/@19.1922063,72.9234965,11z",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
]

print("V5 GENERALIZATION TEST")
print("=" * 70)

for url in v5_generalization_urls:
    test_v5_candidate_url(url)

V5 GENERALIZATION TEST

URL: https://www.nasa.gov/about/
Prediction: Legitimate
Legitimate: 57.67% | Phishing: 42.33%

URL: https://www.ibm.com/support/
Prediction: Legitimate
Legitimate: 83.67% | Phishing: 16.33%

URL: https://www.cisco.com/products/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.intel.com/docs/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.nytimes.com/contact/
Prediction: Legitimate
Legitimate: 86.00% | Phishing: 14.00%

URL: https://google.com/maps/
Prediction: Phishing
Legitimate: 1.33% | Phishing: 98.67%

URL: https://microsoft.com/about/
Prediction: Phishing
Legitimate: 0.33% | Phishing: 99.67%

URL: https://github.com/about/
Prediction: Phishing
Legitimate: 1.67% | Phishing: 98.33%

URL: https://www.google.com/maps/place/
Prediction: Phishing
Legitimate: 28.33% | Phishing: 71.67%

URL: https://www.microsoft.com/en-us/support/
Prediction: Phishing
Legitimate: 30.00% | Phishing: 70.00%

URL: https://

In [49]:
# V6: controlled legitimate structural augmentation

import random
import pandas as pd

random.seed(42)

v6_domains = [
    "google.com",
    "microsoft.com",
    "amazon.com",
    "github.com",
    "wikipedia.org",
    "apple.com",
    "cloudflare.com",
    "mozilla.org",
    "adobe.com",
    "python.org",
    "stackoverflow.com",
    "reddit.com",
    "linkedin.com",
    "dropbox.com",
    "wordpress.com",
    "nasa.gov",
    "ibm.com",
    "cisco.com",
    "intel.com",
    "nytimes.com",
]

v6_paths = [
    # Depth 1
    "/about/",
    "/contact/",
    "/help/",
    "/support/",
    "/docs/",
    "/products/",
    
    # Depth 2
    "/en/about/",
    "/en/contact/",
    "/en/support/",
    "/en/products/",
    "/products/security/",
    "/docs/getting-started/",
    "/support/contact/",
    "/account/settings/",
    
    # Map / coordinate-style structures
    "/maps/@19.1922063,72.9234965,11z",
    "/maps/@40.7128,-74.0060,12z",
    "/maps/@51.5074,-0.1278,11z",
]

v6_urls = []

for domain in v6_domains:
    for path in v6_paths:

        # www version
        v6_urls.append(f"https://www.{domain}{path}")

        # non-www version
        v6_urls.append(f"https://{domain}{path}")


# Keep this controlled rather than creating a huge synthetic dataset
v6_urls = random.sample(
    v6_urls,
    min(500, len(v6_urls))
)

v6_augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in v6_urls]
)

v6_augmented_features["label"] = 1

print("V6 augmentation shape:", v6_augmented_features.shape)

print("\nLabel counts:")
print(v6_augmented_features["label"].value_counts())

print("\nFeature distribution:")
print(
    v6_augmented_features[
        [
            "URLLength",
            "NoOfSubDomain",
            "NoOfDots",
            "NoOfSlashes",
            "PathDepth",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
        ]
    ].describe()
)

V6 augmentation shape: (500, 18)

Label counts:
label
1    500
Name: count, dtype: int64

Feature distribution:
        URLLength  NoOfSubDomain    NoOfDots  NoOfSlashes   PathDepth  \
count  500.000000     500.000000  500.000000   500.000000  500.000000   
mean    35.722000       0.476000    1.836000     4.482000    1.662000   
std      8.057951       0.499924    0.895941     0.500176    0.473502   
min     21.000000       0.000000    1.000000     4.000000    1.000000   
25%     29.000000       0.000000    1.000000     4.000000    1.000000   
50%     34.000000       0.000000    2.000000     4.000000    2.000000   
75%     41.000000       1.000000    2.000000     5.000000    2.000000   
max     58.000000       1.000000    4.000000     5.000000    2.000000   

       NoOfDegitsInURL  NoOfEqualsInURL  NoOfQMarkInURL  
count       500.000000            500.0           500.0  
mean          2.754000              0.0             0.0  
std           6.015468              0.0             0.0 

In [50]:
# V6 candidate training

final_train_df_v6 = pd.concat(
    [
        final_train_df_v4,
        targeted_augmented_features,
        v6_augmented_features,
    ],
    ignore_index=True,
)

X_train_final_v6 = final_train_df_v6[ROBUSTNESS_FEATURES].copy()
y_train_final_v6 = final_train_df_v6["label"].copy()

candidate_model_v6 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v6 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v6),
    ]
)

print("V6 training shape:", final_train_df_v6.shape)

print("\nLabel counts:")
print(y_train_final_v6.value_counts())

print("\nTraining V6 candidate...")

structural_candidate_pipeline_v6.fit(
    X_train_final_v6,
    y_train_final_v6,
)

print("V6 candidate training complete.")

V6 training shape: (388556, 20)

Label counts:
label
1    258528
0    130028
Name: count, dtype: int64

Training V6 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V6 candidate training complete.


In [51]:
# V6 holdout evaluation

v6_holdout_predictions = structural_candidate_pipeline_v6.predict(
    X_holdout_v4
)

print("V6 Holdout Accuracy:")
print(
    structural_candidate_pipeline_v6.score(
        X_holdout_v4,
        y_holdout_v4
    )
)

print("\nV6 Classification Report:")
print(
    classification_report(
        y_holdout_v4,
        v6_holdout_predictions,
        target_names=["Phishing", "Legitimate"],
    )
)

print("\nV6 Confusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v4,
        v6_holdout_predictions,
    )
)

V6 Holdout Accuracy:
0.983941575114703

V6 Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      0.97      0.98     20492
  Legitimate       0.97      1.00      0.99     27022

    accuracy                           0.98     47514
   macro avg       0.99      0.98      0.98     47514
weighted avg       0.98      0.98      0.98     47514


V6 Confusion Matrix:
[[19780   712]
 [   51 26971]]


In [52]:
# V6 targeted robustness tests

def test_v6_candidate_url(url):
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v6.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v6.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    prediction_label = "Phishing" if prediction == 0 else "Legitimate"

    print(f"\nURL: {url}")
    print(f"Prediction: {prediction_label}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )


v6_test_urls = [
    # Legitimate Google Maps structures
    "https://www.google.com/maps/",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",

    # Legitimate deeper paths
    "https://www.google.com/maps/place/",
    "https://www.microsoft.com/en-us/support/",
    "https://www.amazon.com/gp/help/",

    # Legitimate query-heavy URLs
    "https://www.google.com/search?q=artificial+intelligence&page=2",
    "https://www.amazon.com/s?k=cybersecurity+books",
    "https://github.com/search?q=phishing&type=repositories",

    # Legitimate authentication-related URLs
    "https://login.microsoftonline.com/",
    "https://accounts.google.com/",
    "https://www.amazon.com/ap/signin",

    # Synthetic phishing
    "http://secure-login-example.com/verify/",
    "http://secure-account-update-example.com/login/",
    "http://paypal-login-security-example.com/verify/",
    "https://account-verification-update.com/login/",
]

for url in v6_test_urls:
    test_v6_candidate_url(url)


URL: https://www.google.com/maps/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z
Prediction: Legitimate
Legitimate: 95.00% | Phishing: 5.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 3.33% | Phishing: 96.67%

URL: https://www.google.com/maps/place/
Prediction: Legitimate
Legitimate: 93.00% | Phishing: 7.00%

URL: https://www.microsoft.com/en-us/support/
Prediction: Legitimate
Legitimate: 98.33% | Phishing: 1.67%

URL: https://www.amazon.com/gp/help/
Prediction: Legitimate
Legitimate: 65.00% | Phishing: 35.00%

URL: https://www.google.com/search?q=artificial+intelligence&page=2
Prediction: Legitimate
Legitimate: 91.63% | Phishing: 8.37%

URL: https://www.amazon.com/s?k=cybersecurity+books
Prediction: Legitimate
Legitimate: 92.33% | Phishing: 7.67%

URL: https://github.com/search?q=phishing&type=repositories
Prediction: Phishing
Legitimate: 1.67% | Phi

In [54]:
# V6 Google Maps structure diagnostic

v6_maps_test_urls = [
    "https://www.google.com/maps",
    "https://www.google.com/maps/",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser",
]

print("V6 Google Maps diagnostic:\n")

for url in v6_maps_test_urls:
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v6.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v6.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    label = "Phishing" if prediction == 0 else "Legitimate"

    print(f"\nURL: {url}")
    print(f"Prediction: {label}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )

V6 Google Maps diagnostic:


URL: https://www.google.com/maps
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z
Prediction: Legitimate
Legitimate: 95.00% | Phishing: 5.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 3.33% | Phishing: 96.67%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
Prediction: Phishing
Legitimate: 7.00% | Phishing: 93.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser
Prediction: Legitimate
Legitimate: 62.33% | Phishing: 37.67%


In [55]:
# V6 final targeted evaluation

targeted_test_urls_v6 = [
    # Legitimate baseline
    ("Legitimate", "https://www.google.com"),
    ("Legitimate", "https://www.microsoft.com"),
    ("Legitimate", "https://www.amazon.com"),
    ("Legitimate", "https://www.wikipedia.org"),
    ("Legitimate", "https://www.github.com"),

    # Difficult legitimate URLs
    ("Legitimate", "https://www.google.com/search?q=artificial+intelligence+cybersecurity+machine+learning&source=hp&oq=artificial+intelligence"),
    ("Legitimate", "https://www.google.com/maps/search/cybersecurity+companies/@19.0760,72.8777,12z/data=!3m1!4b1!4m5!2m4!5m3!5m2!1s2026-09-13!2s14"),
    ("Legitimate", "https://www.amazon.com/s?k=cybersecurity+books&i=stripbooks&ref=nb_sb_noss"),
    ("Legitimate", "https://www.youtube.com/results?search_query=machine+learning+cybersecurity+tutorial"),
    ("Legitimate", "https://github.com/search?q=phishing+url+detection&type=repositories"),
    ("Legitimate", "https://login.microsoftonline.com/"),
    ("Legitimate", "https://accounts.google.com/"),
    ("Legitimate", "https://www.amazon.com/ap/signin"),
    ("Legitimate", "https://support.google.com/accounts/"),

    # Synthetic phishing
    ("Phishing", "http://secure-login-example.com/verify/account"),
    ("Phishing", "https://account-verification-update.com/login?redirect=%2Faccount%2Fverify"),
    ("Phishing", "http://paypal-login-security-example.com/verify/account"),
    ("Phishing", "http://secure-account-update-example.com/login/verify"),
]

correct = 0
legitimate_total = 0
legitimate_false_positives = 0
phishing_total = 0
phishing_false_negatives = 0

print("V6 Final Targeted Evaluation")
print("=" * 70)

for expected, url in targeted_test_urls_v6:

    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v6.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v6.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]

    predicted = "Phishing" if prediction == 0 else "Legitimate"

    is_correct = predicted == expected

    if is_correct:
        correct += 1

    if expected == "Legitimate":
        legitimate_total += 1
        if predicted != "Legitimate":
            legitimate_false_positives += 1

    else:
        phishing_total += 1
        if predicted != "Phishing":
            phishing_false_negatives += 1

    status = "✓" if is_correct else "✗"

    print(
        f"{status} | Expected: {expected:<11} | "
        f"Predicted: {predicted:<11} | "
        f"Phishing: {phishing_probability:.2%}"
    )
    print(f"    {url}")

print("\n" + "=" * 70)
print(f"Overall correct: {correct}/{len(targeted_test_urls_v6)}")
print(f"Accuracy: {correct / len(targeted_test_urls_v6):.2%}")

print(
    f"Legitimate false positives: "
    f"{legitimate_false_positives}/{legitimate_total}"
)

print(
    f"Phishing false negatives: "
    f"{phishing_false_negatives}/{phishing_total}"
)

V6 Final Targeted Evaluation
✓ | Expected: Legitimate  | Predicted: Legitimate  | Phishing: 0.00%
    https://www.google.com
✓ | Expected: Legitimate  | Predicted: Legitimate  | Phishing: 0.23%
    https://www.microsoft.com
✓ | Expected: Legitimate  | Predicted: Legitimate  | Phishing: 0.00%
    https://www.amazon.com
✓ | Expected: Legitimate  | Predicted: Legitimate  | Phishing: 0.00%
    https://www.wikipedia.org
✓ | Expected: Legitimate  | Predicted: Legitimate  | Phishing: 0.00%
    https://www.github.com
✗ | Expected: Legitimate  | Predicted: Phishing    | Phishing: 60.00%
    https://www.google.com/search?q=artificial+intelligence+cybersecurity+machine+learning&source=hp&oq=artificial+intelligence
✗ | Expected: Legitimate  | Predicted: Phishing    | Phishing: 99.00%
    https://www.google.com/maps/search/cybersecurity+companies/@19.0760,72.8777,12z/data=!3m1!4b1!4m5!2m4!5m3!5m2!1s2026-09-13!2s14
✓ | Expected: Legitimate  | Predicted: Legitimate  | Phishing: 48.67%
    https://www

In [ ]:
# Diagnose V6 legitimate false positives

v6_false_positive_urls = [
    "https://www.google.com/search?q=artificial+intelligence+cybersecurity+machine+learning&source=hp&oq=artificial+intelligence",
    "https://www.google.com/maps/search/cybersecurity+companies/@19.0760,72.8777,12z/data=!3m1!4b1!4m5!2m4!5m3!5m2!1s2026-09-13!2s14",
    "https://www.youtube.com/results?search_query=machine+learning+cybersecurity+tutorial",
    "https://github.com/search?q=phishing+url+detection&type=repositories",
    "https://login.microsoftonline.com/",
    "https://accounts.google.com/",
    "https://support.google.com/accounts/",
]

fp_rows = []

for url in v6_false_positive_urls:
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v6.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v6.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    row = {
        "URL": url,
        "Prediction": "Phishing" if prediction == 0 else "Legitimate",
        "PhishingProbability": round(phishing_probability, 4),
        "LegitimateProbability": round(legitimate_probability, 4),
    }

    row.update(features)
    fp_rows.append(row)

fp_diagnostic_df = pd.DataFrame(fp_rows)

display(
    fp_diagnostic_df[
        [
            "URL",
            "Prediction",
            "PhishingProbability",
            "URLLength",
            "DomainLength",
            "IsDomainIP",
            "TLD",
            "NoOfSubDomain",
            "HasObfuscation",
            "NoOfObfuscatedChar",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "IsHTTPS",
            "NoOfDots",
            "NoOfSlashes",
            "SuspiciousKeywordCount",
            "HasHyphenInDomain",
            "PathDepth",
        ]
    ]
)

In [56]:
# V6 Google Maps structure diagnostic

v6_maps_test_urls = [
    "https://www.google.com/maps",
    "https://www.google.com/maps/",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
    "https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser",
]

print("V6 Google Maps diagnostic:\n")

for url in v6_maps_test_urls:
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v6.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v6.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]
    legitimate_probability = probabilities[classes.index(1)]

    label = "Phishing" if prediction == 0 else "Legitimate"

    print(f"\nURL: {url}")
    print(f"Prediction: {label}")
    print(
        f"Legitimate: {legitimate_probability:.2%} | "
        f"Phishing: {phishing_probability:.2%}"
    )

V6 Google Maps diagnostic:


URL: https://www.google.com/maps
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z
Prediction: Legitimate
Legitimate: 95.00% | Phishing: 5.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu
Prediction: Phishing
Legitimate: 3.33% | Phishing: 96.67%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
Prediction: Phishing
Legitimate: 7.00% | Phishing: 93.00%

URL: https://www.google.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view&source=browser
Prediction: Legitimate
Legitimate: 62.33% | Phishing: 37.67%


In [57]:
# Diagnose V6 legitimate false positives

v6_false_positive_urls = [
    "https://www.google.com/search?q=artificial+intelligence+cybersecurity+machine+learning&source=hp&oq=artificial+intelligence",
    "https://www.google.com/maps/search/cybersecurity+companies/@19.0760,72.8777,12z/data=!3m1!4b1!4m5!2m4!5m3!5m2!1s2026-09-13!2s14",
    "https://www.youtube.com/results?search_query=machine+learning+cybersecurity+tutorial",
    "https://github.com/search?q=phishing+url+detection&type=repositories",
    "https://login.microsoftonline.com/",
    "https://accounts.google.com/",
    "https://support.google.com/accounts/",
]

fp_rows = []

for url in v6_false_positive_urls:
    features = extract_url_features(url)
    input_df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v6.predict(input_df)[0]
    probabilities = structural_candidate_pipeline_v6.predict_proba(input_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]

    row = {
        "URL": url,
        "Prediction": "Phishing" if prediction == 0 else "Legitimate",
        "PhishingProbability": round(phishing_probability, 4),
    }

    row.update(features)
    fp_rows.append(row)

fp_diagnostic_df = pd.DataFrame(fp_rows)

display(
    fp_diagnostic_df[
        [
            "URL",
            "Prediction",
            "PhishingProbability",
            "URLLength",
            "DomainLength",
            "NoOfSubDomain",
            "HasObfuscation",
            "NoOfObfuscatedChar",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "IsHTTPS",
            "NoOfDots",
            "NoOfSlashes",
            "SuspiciousKeywordCount",
            "HasHyphenInDomain",
            "PathDepth",
        ]
    ]
)

,URL,Prediction,PhishingProbability,URLLength,DomainLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
0,https://www.google.com/search?q=artificial+int...,Phishing,0.6000,123,14,1,0,0,0,3,1,1,2,3,0,0,1
1,https://www.google.com/maps/search/cybersecuri...,Phishing,0.9900,127,14,1,0,0,38,1,0,1,4,7,0,0,5
2,https://www.youtube.com/results?search_query=m...,Phishing,0.6400,84,15,1,0,0,0,1,1,1,2,3,0,0,1
3,https://github.com/search?q=phishing+url+detec...,Phishing,0.9800,68,10,0,0,0,0,2,1,1,1,3,0,0,1
4,https://login.microsoftonline.com/,Phishing,0.9657,34,25,1,0,0,0,0,0,1,2,3,1,0,0
5,https://accounts.google.com/,Phishing,0.7890,28,19,1,0,0,0,0,0,1,2,3,1,0,0
6,https://support.google.com/accounts/,Phishing,0.5333,36,18,1,0,0,0,0,0,1,2,4,1,0,1


In [58]:
# Compare legitimate vs phishing URL structures in the cleaned dataset

analysis_df = pd.read_csv("../data/processed/PhiUSIIL_clean.csv")

analysis_df["label_name"] = analysis_df["label"].map({
    0: "Phishing",
    1: "Legitimate"
})

checks = {
    "Contains login": analysis_df["URL"].str.contains("login", case=False, na=False),
    "Contains account": analysis_df["URL"].str.contains("account", case=False, na=False),
    "Contains signin": analysis_df["URL"].str.contains("signin", case=False, na=False),
    "Has query (?)": analysis_df["URL"].str.contains(r"\?", regex=True, na=False),
    "Has equals (=)": analysis_df["URL"].str.contains("=", regex=False, na=False),
    "Path depth >= 1": analysis_df["PathDepth"] >= 1,
    "Path depth >= 2": analysis_df["PathDepth"] >= 2,
    "URL length > 75": analysis_df["URLLength"] > 75,
}

comparison_rows = []

for feature_name, mask in checks.items():

    legitimate = analysis_df["label_name"].eq("Legitimate")
    phishing = analysis_df["label_name"].eq("Phishing")

    legitimate_count = int((mask & legitimate).sum())
    phishing_count = int((mask & phishing).sum())

    legitimate_total = int(legitimate.sum())
    phishing_total = int(phishing.sum())

    comparison_rows.append({
        "Feature": feature_name,
        "Legitimate Count": legitimate_count,
        "Legitimate %": round(
            legitimate_count / legitimate_total * 100, 3
        ),
        "Phishing Count": phishing_count,
        "Phishing %": round(
            phishing_count / phishing_total * 100, 3
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,Feature,Legitimate Count,Legitimate %,Phishing Count,Phishing %
0,Contains login,13,0.010,3433,3.415
1,Contains account,23,0.017,1068,1.062
2,Contains signin,9,0.007,886,0.881
3,Has query (?),0,0.000,6134,6.102
4,Has equals (=),0,0.000,5481,5.453
5,Path depth >= 1,0,0.000,27279,27.138
6,Path depth >= 2,0,0.000,16002,15.919
7,URL length > 75,0,0.000,11084,11.027


In [59]:
# Inspect the legitimate augmentation used by V4/V6

print("V4 better query augmentation:")
display(
    better_query_features_df[
        [
            "URLLength",
            "DomainLength",
            "NoOfSubDomain",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfDots",
            "NoOfSlashes",
            "PathDepth",
            "label",
        ]
    ].describe()
)

print("\nV6 structural augmentation:")
display(
    v6_augmented_features[
        [
            "URLLength",
            "DomainLength",
            "NoOfSubDomain",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfDots",
            "NoOfSlashes",
            "PathDepth",
            "label",
        ]
    ].describe()
)

V4 better query augmentation:


,URLLength,DomainLength,NoOfSubDomain,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfDots,NoOfSlashes,PathDepth,label
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.0,50000.000000,50000.000000,50000.000000,50000.0
mean,74.364560,19.234460,1.158900,13.706000,1.847540,1.0,3.163200,3.551540,1.551540,1.0
std,23.738379,4.811107,0.400417,13.206431,0.652399,0.0,1.437751,0.589704,0.589704,0.0
min,34.000000,9.000000,1.000000,0.000000,1.000000,1.0,2.000000,3.000000,1.000000,1.0
25%,53.000000,16.000000,1.000000,2.000000,1.000000,1.0,2.000000,3.000000,1.000000,1.0
50%,74.000000,19.000000,1.000000,7.000000,2.000000,1.0,2.000000,4.000000,2.000000,1.0
75%,91.000000,22.000000,1.000000,22.000000,2.000000,1.0,5.000000,4.000000,2.000000,1.0
max,143.000000,48.000000,4.000000,52.000000,3.000000,1.0,8.000000,5.000000,3.000000,1.0



V6 structural augmentation:


,URLLength,DomainLength,NoOfSubDomain,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfDots,NoOfSlashes,PathDepth,label
count,500.000000,500.000000,500.000000,500.000000,500.0,500.0,500.000000,500.000000,500.000000,500.0
mean,35.722000,12.722000,0.476000,2.754000,0.0,0.0,1.836000,4.482000,1.662000,1.0
std,8.057951,3.073697,0.499924,6.015468,0.0,0.0,0.895941,0.500176,0.473502,0.0
min,21.000000,7.000000,0.000000,0.000000,0.0,0.0,1.000000,4.000000,1.000000,1.0
25%,29.000000,10.000000,0.000000,0.000000,0.0,0.0,1.000000,4.000000,1.000000,1.0
50%,34.000000,13.000000,0.000000,0.000000,0.0,0.0,2.000000,4.000000,2.000000,1.0
75%,41.000000,15.000000,1.000000,0.000000,0.0,0.0,2.000000,5.000000,2.000000,1.0
max,58.000000,21.000000,1.000000,20.000000,0.0,0.0,4.000000,5.000000,2.000000,1.0


In [60]:
print("Sample legitimate query augmentation URLs:")

display(
    better_query_features_df.head(20)
)

Sample legitimate query augmentation URLs:


,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth,URL,label
0,47,16,0,fr,2,1,1,3,2,1,1,1,2,3,0,0,1,https://www.digitruck.fr/search?q=hello%20world,1
1,48,16,0,com,3,1,0,0,1,2,1,1,2,3,0,0,1,https://www.outplanr.com/search?q=example&page=2,1
2,45,14,0,au,2,2,1,3,2,1,1,1,3,3,0,0,1,https://www.uts.edu.au/search?q=hello%20world,1
3,48,13,0,ae,2,1,0,0,5,2,1,1,2,4,0,0,2,https://www.target.ae/docs/view?id=12345&lang=en,1
4,58,16,0,com,3,1,0,0,0,2,1,1,2,3,0,0,1,https://www.kennykey.com/results?query=example...,1
5,44,18,0,com,3,1,0,0,5,1,1,1,2,3,0,0,1,https://www.reiselykke.com/products?id=12345,1
6,69,22,0,com,3,1,0,0,18,2,1,1,4,3,0,0,1,https://www.worldfootynews.com/location?lat=19...,1
7,49,23,0,com,3,1,0,0,6,1,1,1,2,3,0,0,1,https://www.thenorthendloft.com/article?id=987654,1
8,64,27,0,com,3,1,0,0,0,1,1,1,2,4,1,0,2,https://www.unamericanaincucina.com/account/se...,1
9,46,20,0,com,3,1,0,0,6,1,1,1,2,3,0,0,1,https://www.swapmeetdave.com/article?id=987654,1


In [61]:
# Check whether V6 correctly recognizes its own legitimate query augmentation

query_aug_test = better_query_features_df[
    [
        "URLLength",
        "DomainLength",
        "IsDomainIP",
        "TLD",
        "TLDLength",
        "NoOfSubDomain",
        "HasObfuscation",
        "NoOfObfuscatedChar",
        "NoOfDegitsInURL",
        "NoOfEqualsInURL",
        "NoOfQMarkInURL",
        "IsHTTPS",
        "NoOfDots",
        "NoOfSlashes",
        "SuspiciousKeywordCount",
        "HasHyphenInDomain",
        "PathDepth",
    ]
].copy()

query_predictions = structural_candidate_pipeline_v6.predict(query_aug_test)

query_probabilities = structural_candidate_pipeline_v6.predict_proba(
    query_aug_test
)

classes = list(structural_candidate_pipeline_v6.classes_)
phishing_index = classes.index(0)

print("Total legitimate query examples:", len(query_predictions))
print(
    "Predicted Legitimate:",
    int((query_predictions == 1).sum())
)
print(
    "Predicted Phishing:",
    int((query_predictions == 0).sum())
)

print(
    "Average phishing probability:",
    round(
        query_probabilities[:, phishing_index].mean(),
        4
    )
)

Total legitimate query examples: 50000
Predicted Legitimate: 49981
Predicted Phishing: 19
Average phishing probability: 0.0041


In [62]:
# Compare GitHub search with legitimate query augmentation

github_url = "https://github.com/search?q=phishing+url+detection&type=repositories"

github_features = extract_url_features(github_url)

github_df = pd.DataFrame([github_features])

github_probability = structural_candidate_pipeline_v6.predict_proba(
    github_df
)[0]

classes = list(structural_candidate_pipeline_v6.classes_)

print("GitHub search features:")
display(github_df)

print(
    "GitHub phishing probability:",
    round(github_probability[classes.index(0)], 4)
)

print("\nClosest structural legitimate query examples:")

comparison_columns = [
    "URLLength",
    "DomainLength",
    "NoOfSubDomain",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfDots",
    "NoOfSlashes",
    "PathDepth",
]

github_compare = pd.DataFrame([github_features])[comparison_columns]

display(github_compare)

GitHub search features:


,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
0,68,10,0,com,3,0,0,0,0,2,1,1,1,3,0,0,1


GitHub phishing probability: 0.98

Closest structural legitimate query examples:


,URLLength,DomainLength,NoOfSubDomain,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfDots,NoOfSlashes,PathDepth
0,68,10,0,0,2,1,1,3,1


In [63]:
github_pattern = (
    (better_query_features_df["URLLength"].between(60, 75))
    & (better_query_features_df["NoOfEqualsInURL"] == 2)
    & (better_query_features_df["NoOfQMarkInURL"] == 1)
    & (better_query_features_df["NoOfSlashes"] == 3)
    & (better_query_features_df["PathDepth"] == 1)
    & (better_query_features_df["NoOfDegitsInURL"] == 0)
)

matching_examples = better_query_features_df[github_pattern]

print("Matching legitimate examples:", len(matching_examples))

display(
    matching_examples[
        [
            "URL",
            "URLLength",
            "DomainLength",
            "NoOfSubDomain",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfDots",
            "NoOfSlashes",
            "PathDepth",
            "TLD",
            "label",
        ]
    ].head(20)
)

Matching legitimate examples: 1438


,URL,URLLength,DomainLength,NoOfSubDomain,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfDots,NoOfSlashes,PathDepth,TLD,label
14,https://www.devopsgroup.com/results?query=exam...,61,19,1,0,2,1,2,3,1,com,1
23,https://www.clarionreview.org/results?query=ex...,63,21,1,0,2,1,2,3,1,org,1
67,https://www.maale-adummim.muni.il/results?quer...,67,25,2,0,2,1,3,3,1,il,1
84,https://www.droidafrica.net/results?query=exam...,61,19,1,0,2,1,2,3,1,net,1
98,https://www.stonespecialist.com/results?query=...,65,23,1,0,2,1,2,3,1,com,1
141,https://www.oxfordmindfulness.org/results?quer...,67,25,1,0,2,1,2,3,1,org,1
143,https://www.vecma-toolkit.eu/results?query=exa...,62,20,1,0,2,1,2,3,1,eu,1
148,https://www.cheapcamdenminicab.co.uk/results?q...,70,28,2,0,2,1,3,3,1,uk,1
157,https://www.passingdownthelove.com/results?que...,68,26,1,0,2,1,2,3,1,com,1
177,https://www.petcaretips.net/results?query=exam...,61,19,1,0,2,1,2,3,1,net,1


In [64]:
# Test V6 on legitimate examples structurally closest to GitHub

closest_examples = matching_examples.head(20)

closest_features = closest_examples[ROBUSTNESS_FEATURES].copy()

closest_predictions = structural_candidate_pipeline_v6.predict(
    closest_features
)

closest_probabilities = structural_candidate_pipeline_v6.predict_proba(
    closest_features
)

classes = list(structural_candidate_pipeline_v6.classes_)
phishing_index = classes.index(0)

result = closest_examples[["URL", "TLD"]].copy()
result["Predicted"] = [
    "Phishing" if p == 0 else "Legitimate"
    for p in closest_predictions
]
result["PhishingProbability"] = closest_probabilities[:, phishing_index].round(4)

display(result)

,URL,TLD,Predicted,PhishingProbability
14,https://www.devopsgroup.com/results?query=exam...,com,Legitimate,0.0000
23,https://www.clarionreview.org/results?query=ex...,org,Legitimate,0.0000
67,https://www.maale-adummim.muni.il/results?quer...,il,Legitimate,0.0067
84,https://www.droidafrica.net/results?query=exam...,net,Legitimate,0.0000
98,https://www.stonespecialist.com/results?query=...,com,Legitimate,0.0000
141,https://www.oxfordmindfulness.org/results?quer...,org,Legitimate,0.0000
143,https://www.vecma-toolkit.eu/results?query=exa...,eu,Legitimate,0.0133
148,https://www.cheapcamdenminicab.co.uk/results?q...,uk,Legitimate,0.0000
157,https://www.passingdownthelove.com/results?que...,com,Legitimate,0.0000
177,https://www.petcaretips.net/results?query=exam...,net,Legitimate,0.0000


In [65]:
# Inspect TargetEncoder's learned TLD values

tld_encoder = structural_candidate_pipeline_v6.named_steps[
    "preprocessor"
].named_transformers_["tld"]

print("TLD categories:")
print(tld_encoder.categories_)

print("\nEncoded TLD values:")
print(tld_encoder.encodings_)

TLD categories:
[array(['100', '101', '106', '107', '108', '11', '110', '111', '116', '12',
       '121', '123', '125', '126', '128', '13', '130', '133', '134',
       '136', '14', '140', '145', '146', '148', '149', '15', '150', '151',
       '154', '155', '158', '160', '161', '162', '163', '165', '166',
       '167', '171', '173', '177', '181', '182', '184', '185', '187',
       '189', '196', '198', '199', '20', '200', '203', '206', '210',
       '211', '214', '220', '223', '225', '227', '230', '231', '232',
       '233', '234', '235', '237', '238', '24', '240', '242', '243',
       '250', '252', '254', '26', '28', '30', '38', '39', '42', '43',
       '47', '51', '63', '69', '71', '78', '80', '84', '87', '94', 'ac',
       'academy', 'ad', 'ae', 'aero', 'af', 'africa', 'ag', 'agency',
       'ai', 'al', 'am', 'ao', 'app', 'ar', 'archi', 'art', 'as', 'asia',
       'associates', 'at', 'au', 'audio', 'auto', 'autos', 'aw', 'ax',
       'az', 'ba', 'band', 'bank', 'bar', 'barcelona', 'ba

In [66]:
# Test GitHub structure with different TLD values

github_features = extract_url_features(
    "https://github.com/search?q=phishing+url+detection&type=repositories"
)

test_tlds = ["com", "org", "net", "uk", "il", "eu"]

rows = []

for tld in test_tlds:
    modified = github_features.copy()
    modified["TLD"] = tld

    test_df = pd.DataFrame([modified])

    probabilities = structural_candidate_pipeline_v6.predict_proba(test_df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)

    phishing_probability = probabilities[classes.index(0)]

    rows.append({
        "TLD": tld,
        "PhishingProbability": round(phishing_probability, 4),
        "Prediction": (
            "Phishing"
            if phishing_probability >= 0.5
            else "Legitimate"
        )
    })

display(pd.DataFrame(rows))

,TLD,PhishingProbability,Prediction
0,com,0.9800,Phishing
1,org,0.8767,Phishing
2,net,0.9800,Phishing
3,uk,0.8600,Phishing
4,il,0.8700,Phishing
5,eu,0.9267,Phishing


In [67]:
# One-feature-at-a-time diagnostic for GitHub search

github_url = "https://github.com/search?q=phishing+url+detection&type=repositories"
github_features = extract_url_features(github_url)

base_df = pd.DataFrame([github_features])

classes = list(structural_candidate_pipeline_v6.classes_)
phishing_index = classes.index(0)

base_probability = structural_candidate_pipeline_v6.predict_proba(
    base_df
)[0][phishing_index]

print("Baseline phishing probability:", round(base_probability, 4))

features_to_test = [
    "URLLength",
    "DomainLength",
    "NoOfSubDomain",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfDots",
    "NoOfSlashes",
    "PathDepth",
]

results = []

for feature in features_to_test:
    modified = github_features.copy()

    # Replace one feature with a simpler value
    if feature in ["URLLength", "DomainLength"]:
        modified[feature] = 30 if feature == "URLLength" else 14
    elif feature == "NoOfSubDomain":
        modified[feature] = 1
    elif feature == "NoOfDegitsInURL":
        modified[feature] = 5
    elif feature == "NoOfEqualsInURL":
        modified[feature] = 1
    elif feature == "NoOfQMarkInURL":
        modified[feature] = 0
    elif feature == "NoOfDots":
        modified[feature] = 2
    elif feature == "NoOfSlashes":
        modified[feature] = 2
    elif feature == "PathDepth":
        modified[feature] = 0

    test_df = pd.DataFrame([modified])

    probability = structural_candidate_pipeline_v6.predict_proba(
        test_df
    )[0][phishing_index]

    results.append({
        "ChangedFeature": feature,
        "NewValue": modified[feature],
        "PhishingProbability": round(probability, 4),
    })

display(pd.DataFrame(results))

Baseline phishing probability: 0.98


,ChangedFeature,NewValue,PhishingProbability
0,URLLength,30,0.9633
1,DomainLength,14,0.9833
2,NoOfSubDomain,1,0.4367
3,NoOfDegitsInURL,5,1.0000
4,NoOfEqualsInURL,1,0.9400
5,NoOfQMarkInURL,0,0.9367
6,NoOfDots,2,0.9567
7,NoOfSlashes,2,0.9783
8,PathDepth,0,0.9635


In [68]:
# Compare the same legitimate URL structures with and without www

test_urls = [
    "https://www.google.com/search?q=test&page=2",
    "https://google.com/search?q=test&page=2",

    "https://www.microsoft.com/about/",
    "https://microsoft.com/about/",

    "https://www.github.com/search?q=test&type=repositories",
    "https://github.com/search?q=test&type=repositories",

    "https://www.amazon.com/help/",
    "https://amazon.com/help/",
]

rows = []

for url in test_urls:
    features = extract_url_features(url)
    df = pd.DataFrame([features])

    probabilities = structural_candidate_pipeline_v6.predict_proba(df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)
    phishing_probability = probabilities[classes.index(0)]

    rows.append({
        "URL": url,
        "NoOfSubDomain": features["NoOfSubDomain"],
        "NoOfDots": features["NoOfDots"],
        "NoOfSlashes": features["NoOfSlashes"],
        "PathDepth": features["PathDepth"],
        "PhishingProbability": round(phishing_probability, 4),
        "Prediction": (
            "Phishing"
            if phishing_probability >= 0.5
            else "Legitimate"
        ),
    })

display(pd.DataFrame(rows))

,URL,NoOfSubDomain,NoOfDots,NoOfSlashes,PathDepth,PhishingProbability,Prediction
0,https://www.google.com/search?q=test&page=2,1,2,3,1,0.0800,Legitimate
1,https://google.com/search?q=test&page=2,0,1,3,1,0.9900,Phishing
2,https://www.microsoft.com/about/,1,2,4,1,0.0100,Legitimate
3,https://microsoft.com/about/,0,1,4,1,0.4113,Legitimate
4,https://www.github.com/search?q=test&type=repo...,1,2,3,1,0.0500,Legitimate
5,https://github.com/search?q=test&type=reposito...,0,1,3,1,0.9767,Phishing
6,https://www.amazon.com/help/,1,2,4,1,0.0000,Legitimate
7,https://amazon.com/help/,0,1,4,1,0.1200,Legitimate


In [69]:
# Check non-www representation in current legitimate augmentation

for name, df in [
    ("V4 query augmentation", better_query_features_df),
    ("V6 structural augmentation", v6_augmented_features),
]:
    non_www = df["NoOfSubDomain"] == 0

    print(f"\n{name}")
    print("Total:", len(df))
    print("NoOfSubDomain = 0:", int(non_www.sum()))
    print("NoOfSubDomain > 0:", int((~non_www).sum()))


V4 query augmentation
Total: 50000
NoOfSubDomain = 0: 0
NoOfSubDomain > 0: 50000

V6 structural augmentation
Total: 500
NoOfSubDomain = 0: 262
NoOfSubDomain > 0: 238


In [70]:
# V7: legitimate non-www structural/query augmentation

import random
import pandas as pd

random.seed(42)

v7_domains = [
    "google.com",
    "microsoft.com",
    "amazon.com",
    "github.com",
    "wikipedia.org",
    "apple.com",
    "cloudflare.com",
    "mozilla.org",
    "adobe.com",
    "python.org",
    "stackoverflow.com",
    "reddit.com",
    "linkedin.com",
    "dropbox.com",
    "wordpress.com",
    "nasa.gov",
    "ibm.com",
    "cisco.com",
    "intel.com",
    "nytimes.com",
    "mit.edu",
    "stanford.edu",
    "harvard.edu",
    "bbc.co.uk",
    "ox.ac.uk",
    "europa.eu",
]

v7_templates = [
    # Simple paths
    "/about/",
    "/contact/",
    "/help/",
    "/support/",
    "/docs/",
    "/products/",
    "/services/",
    "/company/",
    "/privacy/",
    "/terms/",
    "/news/",
    "/blog/",

    # Depth 2
    "/en/about/",
    "/en/contact/",
    "/en/support/",
    "/en/products/",
    "/products/security/",
    "/docs/getting-started/",
    "/support/contact/",
    "/account/settings/",

    # Query URLs
    "/search?q=example",
    "/search?q=example&page=2",
    "/search?q=hello%20world",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.8777",
    "/redirect?next=%2Faccount%2Fsettings",
    "/docs?page=2&section=installation",
    "/view?item=123456&source=web&lang=en",

    # Map/coordinate structures
    "/maps/",
    "/maps/@19.1922063,72.9234965,11z",
    "/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
]

v7_urls = [
    f"https://{domain}{template}"
    for domain in v7_domains
    for template in v7_templates
]

v7_urls = random.sample(v7_urls, min(5000, len(v7_urls)))

v7_augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in v7_urls]
)

v7_augmented_features["label"] = 1

print("V7 augmentation shape:", v7_augmented_features.shape)
print("Label counts:")
print(v7_augmented_features["label"].value_counts())

display(
    v7_augmented_features[
        [
            "URLLength",
            "DomainLength",
            "NoOfSubDomain",
            "NoOfDegitsInURL",
            "NoOfEqualsInURL",
            "NoOfQMarkInURL",
            "NoOfDots",
            "NoOfSlashes",
            "PathDepth",
            "label",
        ]
    ].describe()
)

V7 augmentation shape: (936, 18)
Label counts:
label
1    936
Name: count, dtype: int64


,URLLength,DomainLength,NoOfSubDomain,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfDots,NoOfSlashes,PathDepth,label
count,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.0
mean,37.350427,10.461538,0.076923,2.861111,0.611111,0.388889,1.299145,3.944444,1.361111,1.0
std,12.242866,2.241231,0.266612,5.924656,0.859327,0.487759,0.683056,0.705298,0.480579,0.0
min,21.000000,7.000000,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,1.000000,1.0
25%,27.000000,9.000000,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,1.000000,1.0
50%,34.000000,10.000000,0.000000,0.000000,0.000000,0.000000,1.000000,4.000000,1.000000,1.0
75%,46.250000,12.000000,0.000000,2.000000,1.000000,1.000000,1.000000,4.000000,2.000000,1.0
max,77.000000,17.000000,1.000000,20.000000,3.000000,1.000000,4.000000,5.000000,2.000000,1.0


In [71]:
# V7 candidate: V6 + legitimate non-www augmentation

final_train_df_v7 = pd.concat(
    [
        final_train_df_v6,
        v7_augmented_features,
    ],
    ignore_index=True,
)

X_train_final_v7 = final_train_df_v7[ROBUSTNESS_FEATURES].copy()
y_train_final_v7 = final_train_df_v7["label"].copy()

candidate_model_v7 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v7 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v7),
    ]
)

structural_candidate_pipeline_v7.fit(
    X_train_final_v7,
    y_train_final_v7,
)

print("V7 training shape:", final_train_df_v7.shape)
print("\nLabel counts:")
print(y_train_final_v7.value_counts())

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V7 training shape: (389492, 20)

Label counts:
label
1    259464
0    130028
Name: count, dtype: int64


In [72]:
# V7: same 18-case targeted evaluation

targeted_test_urls_v7 = [
    # Legitimate baseline
    ("https://www.google.com", "Legitimate"),
    ("https://www.microsoft.com", "Legitimate"),
    ("https://www.amazon.com", "Legitimate"),
    ("https://www.wikipedia.org", "Legitimate"),
    ("https://www.github.com", "Legitimate"),

    # Legitimate difficult cases
    (
        "https://www.google.com/search?q=artificial+intelligence+cybersecurity+machine+learning&source=hp&oq=artificial+intelligence",
        "Legitimate",
    ),
    (
        "https://www.google.com/maps/search/cybersecurity+companies/@19.0760,72.8777,12z/data=!3m1!4b1!4m5!2m4!5m3!5m2!1s2026-09-13!2s14",
        "Legitimate",
    ),
    (
        "https://www.amazon.com/s?k=cybersecurity+books&i=stripbooks&ref=nb_sb_noss",
        "Legitimate",
    ),
    (
        "https://www.youtube.com/results?search_query=machine+learning+cybersecurity+tutorial",
        "Legitimate",
    ),
    (
        "https://github.com/search?q=phishing+url+detection&type=repositories",
        "Legitimate",
    ),
    (
        "https://login.microsoftonline.com/",
        "Legitimate",
    ),
    (
        "https://accounts.google.com/",
        "Legitimate",
    ),
    (
        "https://www.amazon.com/ap/signin",
        "Legitimate",
    ),
    (
        "https://support.google.com/accounts/",
        "Legitimate",
    ),

    # Synthetic phishing
    (
        "http://secure-login-example.com/verify/account",
        "Phishing",
    ),
    (
        "https://account-verification-update.com/login?redirect=%2Faccount%2Fverify",
        "Phishing",
    ),
    (
        "http://paypal-login-security-example.com/verify/account",
        "Phishing",
    ),
    (
        "http://secure-account-update-example.com/login/verify",
        "Phishing",
    ),
]

rows = []

for url, actual_label in targeted_test_urls_v7:
    features = extract_url_features(url)
    df = pd.DataFrame([features])

    prediction = structural_candidate_pipeline_v7.predict(df)[0]
    probabilities = structural_candidate_pipeline_v7.predict_proba(df)[0]

    classes = list(structural_candidate_pipeline_v7.classes_)
    phishing_probability = probabilities[classes.index(0)]

    predicted_label = (
        "Phishing"
        if prediction == 0
        else "Legitimate"
    )

    rows.append({
        "Actual": actual_label,
        "Predicted": predicted_label,
        "PhishingProbability": round(phishing_probability, 4),
        "Correct": predicted_label == actual_label,
        "URL": url,
    })

v7_targeted_results = pd.DataFrame(rows)

display(v7_targeted_results)

print(
    "\nTargeted accuracy:",
    round(v7_targeted_results["Correct"].mean() * 100, 2),
    "%"
)

print(
    "Legitimate false positives:",
    int(
        (
            (v7_targeted_results["Actual"] == "Legitimate")
            & (v7_targeted_results["Predicted"] == "Phishing")
        ).sum()
    ),
)

print(
    "Phishing false negatives:",
    int(
        (
            (v7_targeted_results["Actual"] == "Phishing")
            & (v7_targeted_results["Predicted"] == "Legitimate")
        ).sum()
    ),
)

,Actual,Predicted,PhishingProbability,Correct,URL
0,Legitimate,Legitimate,0.0000,True,https://www.google.com
1,Legitimate,Legitimate,0.0024,True,https://www.microsoft.com
2,Legitimate,Legitimate,0.0000,True,https://www.amazon.com
3,Legitimate,Legitimate,0.0000,True,https://www.wikipedia.org
4,Legitimate,Legitimate,0.0000,True,https://www.github.com
5,Legitimate,Phishing,0.6533,False,https://www.google.com/search?q=artificial+int...
6,Legitimate,Phishing,0.9867,False,https://www.google.com/maps/search/cybersecuri...
7,Legitimate,Phishing,0.5100,False,https://www.amazon.com/s?k=cybersecurity+books...
8,Legitimate,Phishing,0.7033,False,https://www.youtube.com/results?search_query=m...
9,Legitimate,Phishing,0.5467,False,https://github.com/search?q=phishing+url+detec...



Targeted accuracy: 55.56 %
Legitimate false positives: 8
Phishing false negatives: 0


In [73]:
# V7 holdout evaluation

v7_holdout_predictions = structural_candidate_pipeline_v7.predict(
    X_holdout_v4
)

v7_holdout_probabilities = structural_candidate_pipeline_v7.predict_proba(
    X_holdout_v4
)

print("V7 Holdout Accuracy:")
print(
    (v7_holdout_predictions == y_holdout_v4).mean()
)

print("\nV7 Classification Report:")
print(
    classification_report(
        y_holdout_v4,
        v7_holdout_predictions,
        target_names=["Phishing", "Legitimate"],
    )
)

print("\nV7 Confusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v4,
        v7_holdout_predictions,
    )
)

V7 Holdout Accuracy:
0.9843835501115461

V7 Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      0.97      0.98     20492
  Legitimate       0.98      1.00      0.99     27022

    accuracy                           0.98     47514
   macro avg       0.99      0.98      0.98     47514
weighted avg       0.98      0.98      0.98     47514


V7 Confusion Matrix:
[[19806   686]
 [   56 26966]]


In [74]:
# Test a "www-neutral" version of the subdomain feature
# WITHOUT changing the actual feature_extractor.py yet.

test_urls = [
    "https://www.google.com/search?q=test&page=2",
    "https://google.com/search?q=test&page=2",
    "https://www.github.com/search?q=test&type=repositories",
    "https://github.com/search?q=test&type=repositories",
    "https://www.microsoft.com/about/",
    "https://microsoft.com/about/",
    "https://www.amazon.com/help/",
    "https://amazon.com/help/",
]

rows = []

for url in test_urls:
    features = extract_url_features(url)

    # Simulate ignoring the conventional "www" prefix
    hostname = urlparse(url).hostname or ""

    if hostname.startswith("www."):
        features["NoOfSubDomain"] = max(
            0,
            features["NoOfSubDomain"] - 1
        )

    df = pd.DataFrame([features])

    probabilities = structural_candidate_pipeline_v6.predict_proba(df)[0]
    classes = list(structural_candidate_pipeline_v6.classes_)
    phishing_probability = probabilities[classes.index(0)]

    rows.append({
        "URL": url,
        "NoOfSubDomainAfterNormalization": features["NoOfSubDomain"],
        "NoOfDots": features["NoOfDots"],
        "PhishingProbability": round(phishing_probability, 4),
        "Prediction": (
            "Phishing"
            if phishing_probability >= 0.5
            else "Legitimate"
        ),
    })

display(pd.DataFrame(rows))

,URL,NoOfSubDomainAfterNormalization,NoOfDots,PhishingProbability,Prediction
0,https://www.google.com/search?q=test&page=2,0,2,0.9800,Phishing
1,https://google.com/search?q=test&page=2,0,1,0.9967,Phishing
2,https://www.github.com/search?q=test&type=repo...,0,2,0.9533,Phishing
3,https://github.com/search?q=test&type=reposito...,0,1,0.9667,Phishing
4,https://www.microsoft.com/about/,0,2,0.9300,Phishing
5,https://microsoft.com/about/,0,1,0.7553,Phishing
6,https://www.amazon.com/help/,0,2,0.9367,Phishing
7,https://amazon.com/help/,0,1,0.8267,Phishing


In [75]:
# Compare difficult real URLs with the legitimate V6 augmentation
# using the 17 production features.

difficult_urls = [
    "https://www.google.com/search?q=artificial+intelligence+cybersecurity+machine+learning&source=hp&oq=artificial+intelligence",
    "https://www.google.com/maps/search/cybersecurity+companies/@19.0760,72.8777,12z/data=!3m1!4b1!4m5!2m4!5m3!5m2!1s2026-09-13!2s14",
    "https://www.amazon.com/s?k=cybersecurity+books&i=stripbooks&ref=nb_sb_noss",
    "https://www.youtube.com/results?search_query=machine+learning+cybersecurity+tutorial",
    "https://github.com/search?q=phishing+url+detection&type=repositories",
]

difficult_features = pd.DataFrame(
    [extract_url_features(url) for url in difficult_urls]
)

print("DIFFICULT REAL URLs")
display(
    difficult_features[
        ROBUSTNESS_FEATURES
    ]
)

print("\nV6 LEGITIMATE QUERY AUGMENTATION")
display(
    better_query_features_df[
        ROBUSTNESS_FEATURES
    ].describe()
)

DIFFICULT REAL URLs


,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
0,123,14,0,com,3,1,0,0,0,3,1,1,2,3,0,0,1
1,127,14,0,com,3,1,0,0,38,1,0,1,4,7,0,0,5
2,74,14,0,com,3,1,0,0,0,3,1,1,2,3,0,0,1
3,84,15,0,com,3,1,0,0,0,1,1,1,2,3,0,0,1
4,68,10,0,com,3,0,0,0,0,2,1,1,1,3,0,0,1



V6 LEGITIMATE QUERY AUGMENTATION


,URLLength,DomainLength,IsDomainIP,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
count,50000.000000,50000.000000,50000.0,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.0,50000.0,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,74.364560,19.234460,0.0,2.727700,1.158900,0.349080,1.956720,13.706000,1.847540,1.0,1.0,3.163200,3.551540,0.152580,0.075160,1.551540
std,23.738379,4.811107,0.0,0.514663,0.400417,0.476684,2.902043,13.206431,0.652399,0.0,0.0,1.437751,0.589704,0.359753,0.263652,0.589704
min,34.000000,9.000000,0.0,2.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.0,1.0,2.000000,3.000000,0.000000,0.000000,1.000000
25%,53.000000,16.000000,0.0,2.000000,1.000000,0.000000,0.000000,2.000000,1.000000,1.0,1.0,2.000000,3.000000,0.000000,0.000000,1.000000
50%,74.000000,19.000000,0.0,3.000000,1.000000,0.000000,0.000000,7.000000,2.000000,1.0,1.0,2.000000,4.000000,0.000000,0.000000,2.000000
75%,91.000000,22.000000,0.0,3.000000,1.000000,1.000000,6.000000,22.000000,2.000000,1.0,1.0,5.000000,4.000000,0.000000,0.000000,2.000000
max,143.000000,48.000000,0.0,13.000000,4.000000,1.000000,9.000000,52.000000,3.000000,1.0,1.0,8.000000,5.000000,2.000000,1.000000,3.000000


In [76]:
test_urls = [
    "https://www.google.com/search?q=test&page=2",
    "https://www.google.com/maps/search/test/@19.0760,72.8777,12z/data=!3m1!4b1",
    "https://github.com/search?q=phishing&type=repositories",
]

for url in test_urls:
    print("\nURL:", url)
    print(extract_url_features(url))


URL: https://www.google.com/search?q=test&page=2
{'URLLength': 43, 'DomainLength': 14, 'IsDomainIP': 0, 'TLD': 'com', 'TLDLength': 3, 'NoOfSubDomain': 1, 'HasObfuscation': 0, 'NoOfObfuscatedChar': 0, 'NoOfDegitsInURL': 1, 'NoOfEqualsInURL': 2, 'NoOfQMarkInURL': 1, 'IsHTTPS': 1, 'NoOfDots': 2, 'NoOfSlashes': 3, 'SuspiciousKeywordCount': 0, 'HasHyphenInDomain': 0, 'PathDepth': 1}

URL: https://www.google.com/maps/search/test/@19.0760,72.8777,12z/data=!3m1!4b1
{'URLLength': 74, 'DomainLength': 14, 'IsDomainIP': 0, 'TLD': 'com', 'TLDLength': 3, 'NoOfSubDomain': 1, 'HasObfuscation': 0, 'NoOfObfuscatedChar': 0, 'NoOfDegitsInURL': 18, 'NoOfEqualsInURL': 1, 'NoOfQMarkInURL': 0, 'IsHTTPS': 1, 'NoOfDots': 4, 'NoOfSlashes': 7, 'SuspiciousKeywordCount': 0, 'HasHyphenInDomain': 0, 'PathDepth': 5}

URL: https://github.com/search?q=phishing&type=repositories
{'URLLength': 54, 'DomainLength': 10, 'IsDomainIP': 0, 'TLD': 'com', 'TLDLength': 3, 'NoOfSubDomain': 0, 'HasObfuscation': 0, 'NoOfObfuscatedCh

In [78]:
import joblib

model = joblib.load("../models/trustlens_production_pipeline.joblib")

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


In [79]:
url = "https://github.com/search?q=phishing&type=repositories"

features = extract_url_features(url)
baseline = pd.DataFrame([features])

print("BASELINE")
print(model.predict_proba(baseline)[0])

BASELINE
[0.99666667 0.00333333]


In [89]:
# =========================================================
# RECOVERY STEP 6 - SAVE + RELOAD FINAL V4 MODEL
# =========================================================

import joblib
from pathlib import Path
import pandas as pd

production_model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

# Save V4
joblib.dump(
    structural_candidate_pipeline_v4,
    production_model_path
)

print("Saved V4 production model to:", production_model_path)

# Reload from disk
reloaded_v4 = joblib.load(
    production_model_path
)

print("Reload successful.")
print("Classes:", reloaded_v4.classes_)

# ---------------------------------------------------------
# VERIFY REAL LONG LEGITIMATE URL
# ---------------------------------------------------------

real_long_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

features = pd.DataFrame(
    [extract_url_features(real_long_url)]
)[ROBUSTNESS_FEATURES]

prediction = reloaded_v4.predict(features)[0]
probabilities = reloaded_v4.predict_proba(features)[0]

probability_map = dict(
    zip(
        reloaded_v4.classes_,
        probabilities,
    )
)

print("\nREAL LONG URL TEST")
print(
    "Prediction:",
    "Legitimate" if prediction == 1 else "Phishing"
)
print(
    f"Legitimate: {probability_map.get(1, 0) * 100:.2f}%"
    f" | Phishing: {probability_map.get(0, 0) * 100:.2f}%"
)

# ---------------------------------------------------------
# VERIFY PHISHING QUERY URL
# ---------------------------------------------------------

phishing_test_url = (
    "https://account-verification-update.com/"
    "login?redirect=%2Faccount%2Fverify"
)

features = pd.DataFrame(
    [extract_url_features(phishing_test_url)]
)[ROBUSTNESS_FEATURES]

prediction = reloaded_v4.predict(features)[0]
probabilities = reloaded_v4.predict_proba(features)[0]

probability_map = dict(
    zip(
        reloaded_v4.classes_,
        probabilities,
    )
)

print("\nPHISHING QUERY TEST")
print(
    "Prediction:",
    "Legitimate" if prediction == 1 else "Phishing"
)
print(
    f"Legitimate: {probability_map.get(1, 0) * 100:.2f}%"
    f" | Phishing: {probability_map.get(0, 0) * 100:.2f}%"
)

Saved V4 production model to: ..\models\trustlens_production_pipeline.joblib


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


Reload successful.
Classes: [0 1]

REAL LONG URL TEST
Prediction: Legitimate
Legitimate: 79.67% | Phishing: 20.33%

PHISHING QUERY TEST
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%


In [80]:
url = "https://github.com/search?q=phishing&type=repositories"

features = extract_url_features(url)
baseline = pd.DataFrame([features])

print("BASELINE")
print("Features:", features)
print("Probabilities:", model.predict_proba(baseline)[0])

BASELINE
Features: {'URLLength': 54, 'DomainLength': 10, 'IsDomainIP': 0, 'TLD': 'com', 'TLDLength': 3, 'NoOfSubDomain': 0, 'HasObfuscation': 0, 'NoOfObfuscatedChar': 0, 'NoOfDegitsInURL': 0, 'NoOfEqualsInURL': 2, 'NoOfQMarkInURL': 1, 'IsHTTPS': 1, 'NoOfDots': 1, 'NoOfSlashes': 3, 'SuspiciousKeywordCount': 0, 'HasHyphenInDomain': 0, 'PathDepth': 1}
Probabilities: [0.99666667 0.00333333]


In [81]:
test = baseline.copy()

tests = {
    "baseline": {},
    "subdomain_1": {"NoOfSubDomain": 1},
    "equals_1": {"NoOfEqualsInURL": 1},
    "qmark_0": {"NoOfQMarkInURL": 0},
    "dots_2": {"NoOfDots": 2},
    "slashes_2": {"NoOfSlashes": 2},
    "pathdepth_0": {"PathDepth": 0},
    "url_length_40": {"URLLength": 40},
}

for name, changes in tests.items():
    modified = test.copy()

    for feature, value in changes.items():
        modified[feature] = value

    prob = model.predict_proba(modified)[0]

    print(
        f"{name:15} -> "
        f"Legitimate: {prob[1]*100:.2f}% | "
        f"Phishing: {prob[0]*100:.2f}%"
    )

baseline        -> Legitimate: 0.33% | Phishing: 99.67%
subdomain_1     -> Legitimate: 68.00% | Phishing: 32.00%
equals_1        -> Legitimate: 8.67% | Phishing: 91.33%
qmark_0         -> Legitimate: 4.33% | Phishing: 95.67%
dots_2          -> Legitimate: 6.67% | Phishing: 93.33%
slashes_2       -> Legitimate: 2.17% | Phishing: 97.83%
pathdepth_0     -> Legitimate: 5.14% | Phishing: 94.86%
url_length_40   -> Legitimate: 0.67% | Phishing: 99.33%


In [82]:
tests = {
    "baseline": {
        "NoOfSubDomain": 0,
        "NoOfDots": 1,
        "NoOfEqualsInURL": 2,
        "NoOfQMarkInURL": 1,
        "NoOfSlashes": 3,
        "PathDepth": 1,
    },
    "subdomain_1": {
        "NoOfSubDomain": 1,
        "NoOfDots": 1,
        "NoOfEqualsInURL": 2,
        "NoOfQMarkInURL": 1,
        "NoOfSlashes": 3,
        "PathDepth": 1,
    },
    "subdomain_1_dots_2": {
        "NoOfSubDomain": 1,
        "NoOfDots": 2,
        "NoOfEqualsInURL": 2,
        "NoOfQMarkInURL": 1,
        "NoOfSlashes": 3,
        "PathDepth": 1,
    },
}

for name, changes in tests.items():
    modified = baseline.copy()

    for feature, value in changes.items():
        modified[feature] = value

    prob = model.predict_proba(modified)[0]

    print(
        f"{name:25} -> "
        f"Legitimate: {prob[1]*100:.2f}% | "
        f"Phishing: {prob[0]*100:.2f}%"
    )

baseline                  -> Legitimate: 0.33% | Phishing: 99.67%
subdomain_1               -> Legitimate: 68.00% | Phishing: 32.00%
subdomain_1_dots_2        -> Legitimate: 98.67% | Phishing: 1.33%


In [83]:
test_urls = [
    "https://google.com/search?q=test&page=2",
    "https://microsoft.com/search?q=test&page=2",
    "https://amazon.com/s?k=laptop&page=2",
    "https://github.com/search?q=phishing&type=repositories",
    "https://wikipedia.org/w/index.php?search=test",
    "https://apple.com/search?q=iphone&page=2",
]

for url in test_urls:
    features = extract_url_features(url)
    df = pd.DataFrame([features])
    prob = model.predict_proba(df)[0]

    print(
        f"{url}\n"
        f"  subdomain={features['NoOfSubDomain']}, "
        f"dots={features['NoOfDots']}, "
        f"equals={features['NoOfEqualsInURL']}, "
        f"qmark={features['NoOfQMarkInURL']}, "
        f"slashes={features['NoOfSlashes']}, "
        f"path={features['PathDepth']}\n"
        f"  Legitimate: {prob[1]*100:.2f}% | "
        f"Phishing: {prob[0]*100:.2f}%\n"
    )

https://google.com/search?q=test&page=2
  subdomain=0, dots=1, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 1.33% | Phishing: 98.67%

https://microsoft.com/search?q=test&page=2
  subdomain=0, dots=1, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 0.33% | Phishing: 99.67%

https://amazon.com/s?k=laptop&page=2
  subdomain=0, dots=1, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 1.67% | Phishing: 98.33%

https://github.com/search?q=phishing&type=repositories
  subdomain=0, dots=1, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 0.33% | Phishing: 99.67%

https://wikipedia.org/w/index.php?search=test
  subdomain=0, dots=2, equals=1, qmark=1, slashes=4, path=2
  Legitimate: 8.00% | Phishing: 92.00%

https://apple.com/search?q=iphone&page=2
  subdomain=0, dots=1, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 1.67% | Phishing: 98.33%



In [84]:
test_urls = [
    "https://www.google.com/search?q=test&page=2",
    "https://www.microsoft.com/search?q=test&page=2",
    "https://www.amazon.com/s?k=laptop&page=2",
    "https://www.github.com/search?q=phishing&type=repositories",
    "https://www.apple.com/search?q=iphone&page=2",
]

for url in test_urls:
    features = extract_url_features(url)
    df = pd.DataFrame([features])
    prob = model.predict_proba(df)[0]

    print(
        f"{url}\n"
        f"  subdomain={features['NoOfSubDomain']}, "
        f"dots={features['NoOfDots']}, "
        f"equals={features['NoOfEqualsInURL']}, "
        f"qmark={features['NoOfQMarkInURL']}, "
        f"slashes={features['NoOfSlashes']}, "
        f"path={features['PathDepth']}\n"
        f"  Legitimate: {prob[1]*100:.2f}% | "
        f"Phishing: {prob[0]*100:.2f}%\n"
    )

https://www.google.com/search?q=test&page=2
  subdomain=1, dots=2, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 89.00% | Phishing: 11.00%

https://www.microsoft.com/search?q=test&page=2
  subdomain=1, dots=2, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 89.33% | Phishing: 10.67%

https://www.amazon.com/s?k=laptop&page=2
  subdomain=1, dots=2, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 90.33% | Phishing: 9.67%

https://www.github.com/search?q=phishing&type=repositories
  subdomain=1, dots=2, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 99.33% | Phishing: 0.67%

https://www.apple.com/search?q=iphone&page=2
  subdomain=1, dots=2, equals=2, qmark=1, slashes=3, path=1
  Legitimate: 98.67% | Phishing: 1.33%



In [ ]:
# =========================================================
# RECOVERY STEP 7 - VERIFY SAVED MODEL + FEATURES
# =========================================================

print("MODEL OBJECT")
print(type(reloaded_v4))

print("\nMODEL CLASSES")
print(reloaded_v4.classes_)

print("\nROBUSTNESS FEATURES")
print(ROBUSTNESS_FEATURES)

print("\nEXTRACTED FEATURES FOR GOOGLE MAPS")
google_features = extract_url_features(real_long_url)

for feature in ROBUSTNESS_FEATURES:
    print(f"{feature}: {google_features[feature]}")

print("\nFINAL PREDICTION")
google_df = pd.DataFrame([google_features])[ROBUSTNESS_FEATURES]

print("Prediction:", reloaded_v4.predict(google_df)[0])
print("Probabilities:", reloaded_v4.predict_proba(google_df)[0])

MODEL OBJECT


NameError: name 'reloaded_v4' is not defined